[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_10/MFM_ch10_seminar/MFM_ch10_seminar.ipynb)

# MFM_ch10_seminar

Seminar 10 computations: moments, bias and standard error of the Roll estimator, Glosten-Milgrom for any prior and learning speed, the Kyle equilibrium and lambda as a regression slope, Almgren-Chriss by Euler-Lagrange, the PIN likelihood in stable form; intraday U-shape with day-block bootstrap, spread estimators at two frequencies, spread and Amihud measures across 25 assets, illiquidity and the VIX with HAC errors, the volume-price relation with an instrumental variable, the Bitcoin clock, price discovery between the BET ETF and BET-TR, and the time-series illiquidity premium on the BVB, US and crypto markets.

*Modelling Financial Markets (MFM), Chapter 10: Market Microstructure. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Gray     = '#7F7F7F'   # doar linii de referinta
GROUP_COL = {'US': MainBlue, 'BVB': IDAred, 'Crypto': Amber}

HERE = '.'
SEED = 42
START2 = '2024-09-19'
START10 = '2016-09-19'
B_BOOT = 1000


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, labels, ncol=3, y=0.0):
    """O singura legenda pentru o figura cu mai multe panouri, sub figura."""
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# cheie -> (simbol, eticheta, grup)
ASSETS = {
    'SPY': ('SPY.US', 'SPY', 'US'), 'AAPL': ('AAPL.US', 'Apple', 'US'), 'MSFT': ('MSFT.US', 'Microsoft', 'US'),
    'NVDA': ('NVDA.US', 'NVIDIA', 'US'), 'JPM': ('JPM.US', 'JPMorgan', 'US'), 'GME': ('GME.US', 'GameStop', 'US'),
    'MSTR': ('MSTR.US', 'Strategy', 'US'),
    'TLV': ('TLV.RO', 'Banca Transilvania', 'BVB'), 'SNP': ('SNP.RO', 'OMV Petrom', 'BVB'),
    'H2O': ('H2O.RO', 'Hidroelectrica', 'BVB'), 'BRD': ('BRD.RO', 'BRD', 'BVB'), 'SNG': ('SNG.RO', 'Romgaz', 'BVB'),
    'SNN': ('SNN.RO', 'Nuclearelectrica', 'BVB'), 'TGN': ('TGN.RO', 'Transgaz', 'BVB'),
    'EL': ('EL.RO', 'Electrica', 'BVB'), 'DIGI': ('DIGI.RO', 'Digi', 'BVB'), 'TEL': ('TEL.RO', 'Transelectrica', 'BVB'),
    'TVBETETF': ('TVBETETF.RO', 'BET ETF', 'BVB'),
    'BTC': ('BTC-USD.CC', 'Bitcoin', 'Crypto'), 'ETH': ('ETH-USD.CC', 'Ethereum', 'Crypto'),
    'SOL': ('SOL-USD.CC', 'Solana', 'Crypto'), 'XRP': ('XRP-USD.CC', 'XRP', 'Crypto'),
    'DOGE': ('DOGE-USD.CC', 'Dogecoin', 'Crypto'), 'ADA': ('ADA-USD.CC', 'Cardano', 'Crypto'),
    'LTC': ('LTC-USD.CC', 'Litecoin', 'Crypto'),
}
GROUPS = {g: [k for k, v in ASSETS.items() if v[2] == g] for g in ('US', 'BVB', 'Crypto')}
LABELS = {k: v[1] for k, v in ASSETS.items()}
# evenimente de ajustare mari care sunt dividende in numerar, nu split-uri (verificate la emitent):
#   SNN.RO 2018-12-21: dividend brut de 1,61 lei pe actiune, data ex 21.12.2018
#   (https://www.nuclearelectrica.ro/wp-content/uploads/2018/12/SNN_Comunicat-plata-dividende-suplimentare_EN.pdf)
CASH_DIVIDENDS = {'SNN.RO': ['2018-12-21']}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    if symbol in _CACHE:
        return _CACHE[symbol]
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    _CACHE[symbol] = pd.read_csv(src, index_col='date', parse_dates=True).sort_index()
    return _CACHE[symbol]


def read_reference_rate(currency='USD', start='2014-01-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = ('REF', currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'rate']).drop_duplicates('date').set_index('date')['rate']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def _clean(key, d):
    d = d[(d['high'] > d['low']) & (d['volume'].fillna(0) > 0)]
    if ASSETS[key][2] != 'Crypto':
        d = d[d.index.dayofweek < 5]
        # cotatii eronate izolate: pretul ajustat de peste doua ori mai mare / mai mic decat mediana locala (21 de zile)
        med = d['adjusted_close'].rolling(21, center=True, min_periods=5).median()
        d = d[np.abs(np.log(d['adjusted_close'] / med)) < np.log(2)]
    return d


def ohlc(key, start='2016-09-19', end=END):
    """Deschidere, maxim, minim, inchidere ajustate cu factorul pretului ajustat (dividende si split-uri)."""
    d = read_market(ASSETS[key][0]).loc[start:end].copy()
    f = d['adjusted_close'] / d['close']
    for c in ['open', 'high', 'low', 'close']:
        d[c] = d[c] * f
    return _clean(key, d)[['open', 'high', 'low', 'close', 'volume']]


def split_adjusted_close(d, symbol=None):
    """Pretul de inchidere ajustat DOAR pentru split-uri si actiuni gratuite (aceeasi baza ca volumul in numar de
    actiuni); dividendele in numerar mari din CASH_DIVIDENDS nu sunt tratate ca split-uri."""
    k = (d['close'].shift(1) / d['close']) / (d['adjusted_close'].shift(1) / d['adjusted_close'])
    k = k.where(np.abs(np.log(k)) > 0.15, 1.0).fillna(1.0)      # zilele de split: raportul de divizare
    for day in CASH_DIVIDENDS.get(symbol, []):
        k.loc[k.index == pd.Timestamp(day)] = 1.0
    back = k[::-1].cumprod()[::-1].shift(-1).fillna(1.0)          # produsul split-urilor ulterioare fiecarei zile
    return d['close'] / back


def returns(key, start='2016-09-19', end=END):
    """Randamente log zilnice din pretul ajustat (zilele fara tranzactii eliminate)."""
    d = read_market(ASSETS[key][0]).loc[start:end]
    d = _clean(key, d)
    return np.log(d['adjusted_close']).diff().dropna().rename(key)


def dollar_volume(key, start='2016-09-19', end=END):
    """Valoarea tranzactionata zilnic, in USD."""
    d = read_market(ASSETS[key][0]).loc[:end]
    grp = ASSETS[key][2]
    if grp == 'Crypto':
        dv = d['volume']                                          # deja in USD
    else:
        dv = split_adjusted_close(d, ASSETS[key][0]) * d['volume']
        if grp == 'BVB':                                          # RON -> USD la cursul de referinta BNR
            fx = read_reference_rate('USD', start='2014-01-01', end=end)
            dv = dv / fx.reindex(dv.index).ffill()
    d = _clean(key, d.assign(dv=dv))
    return d['dv'].loc[start:end].dropna().rename(key)


def intraday_spy():
    """Bare de 5 minute SPY, sesiunea regulata (09:30-15:55 = inceputul barei), doar zilele complete (78 de bare)."""
    fname = 'intraday/SPY.US_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    t = d['datetime_utc'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)
    d = d.assign(t=t, date=t.dt.normalize(), tod=t.dt.strftime('%H:%M')).drop(columns='datetime_utc')
    d = d[(d['tod'] >= '09:30') & (d['tod'] <= '15:55')]
    n = d.groupby('date').size()
    d = d[d['date'].isin(n[n == 78].index)].set_index('t').sort_index()
    d['r'] = np.log(d['close']).groupby(d['date']).diff()
    first = d['tod'] == '09:30'
    d.loc[first, 'r'] = np.log(d.loc[first, 'close'] / d.loc[first, 'open'])   # prima bara: de la deschidere
    d['dv'] = d['close'] * d['volume']
    return d


def intraday_btc():
    """Bare de 5 minute Bitcoin (UTC), 24 de ore din 24; volumul nu este folosit (lipseste pe multe bare)."""
    fname = 'intraday/BTC-USD.CC_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    d = d.set_index('datetime_utc').sort_index()
    d = d[~d.index.duplicated()]
    d['date'] = d.index.normalize()
    d['r'] = np.log(d['close']).diff()
    gap = d.index.to_series().diff() != pd.Timedelta('5min')
    d.loc[gap, 'r'] = np.nan                                       # fara randamente peste goluri
    return d

## Estimators and models

In [ ]:
def roll_spread(close):
    """Spread-ul relativ Roll din covarianta de ordinul 1 a randamentelor log; NaN daca covarianta este pozitiva."""
    r = np.log(np.asarray(close, float))
    dp = np.diff(r)
    c = np.mean((dp[1:] - dp.mean()) * (dp[:-1] - dp.mean()))
    return (2 * np.sqrt(-c) if c < 0 else np.nan), c


def cs_spread(high, low, close=None):
    """Estimatorul Corwin-Schultz pentru fiecare pereche de perioade consecutive (valorile negative devin 0).
    Cu close (date zilnice): ajustarea pentru miscarea de peste noapte din Corwin si Schultz (2012): daca minimul zilei t+1
    este peste inchiderea zilei t, maximul si minimul lui t+1 scad cu diferenta; daca maximul lui t+1 este sub
    inchiderea lui t, ambele cresc cu diferenta. Barele intraday ale aceleiasi sesiuni nu se ajusteaza."""
    H, L = np.asarray(high, float), np.asarray(low, float)
    h0, l0, h1, l1 = H[:-1], L[:-1], H[1:].copy(), L[1:].copy()
    if close is not None:
        c0 = np.asarray(close, float)[:-1]
        up = l1 > c0
        d = np.where(up, l1 - c0, 0.0)
        h1, l1 = h1 - d, l1 - d
        dn = h1 < c0
        d = np.where(dn, c0 - h1, 0.0)
        h1, l1 = h1 + d, l1 + d
    beta = np.log(h0 / l0) ** 2 + np.log(h1 / l1) ** 2
    gamma = np.log(np.maximum(h0, h1) / np.minimum(l0, l1)) ** 2
    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    return np.clip(2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha)), 0, None)


def ar_terms(close, high, low):
    """Termenii Abdi-Ranaldo: 4 (c_t - eta_t)(c_t - eta_{t+1}); media lor estimeaza s^2."""
    c = np.log(np.asarray(close, float))
    eta = (np.log(np.asarray(high, float)) + np.log(np.asarray(low, float))) / 2
    return 4 * (c[:-1] - eta[:-1]) * (c[:-1] - eta[1:])


def ar_spread(close, high, low):
    """Spread-ul relativ Abdi-Ranaldo: sqrt(max(media termenilor, 0))."""
    m = np.mean(ar_terms(close, high, low))
    return np.sqrt(max(m, 0.0)), m


def edge_spread(open_, high, low, close, sign=True):
    """EDGE (Ardia, Guidotti si Kroencke, 2024, JFE 161, 103916): spread-ul relativ din preturile OHLC.
    Transcrierea implementarii de referinta a autorilor (pachetul bidask, licenta MIT). sign=True intoarce estimarea
    cu semn; pentru medii pe mai multe ferestre, autorii recomanda estimarile cu semn cu valorile negative puse la zero."""
    o, h, l, c = (np.log(np.asarray(x, float)) for x in (open_, high, low, close))
    if len(o) < 3:
        return np.nan
    m = (h + l) / 2.
    h1, l1, c1, m1 = h[:-1], l[:-1], c[:-1], m[:-1]
    o, h, l, c, m = o[1:], h[1:], l[1:], c[1:], m[1:]
    r1, r2, r3, r4, r5 = m - o, o - m1, m - c1, c1 - m1, o - c1
    tau = np.where(np.isnan(h) | np.isnan(l) | np.isnan(c1), np.nan, (h != l) | (l != c1))
    po1 = tau * np.where(np.isnan(o) | np.isnan(h), np.nan, o != h)
    po2 = tau * np.where(np.isnan(o) | np.isnan(l), np.nan, o != l)
    pc1 = tau * np.where(np.isnan(c1) | np.isnan(h1), np.nan, c1 != h1)
    pc2 = tau * np.where(np.isnan(c1) | np.isnan(l1), np.nan, c1 != l1)
    pt = np.nanmean(tau)
    po = np.nanmean(po1) + np.nanmean(po2)
    pc = np.nanmean(pc1) + np.nanmean(pc2)
    if np.nansum(tau) < 2 or po == 0 or pc == 0:
        return np.nan
    d1 = r1 - np.nanmean(r1) / pt * tau
    d3 = r3 - np.nanmean(r3) / pt * tau
    d5 = r5 - np.nanmean(r5) / pt * tau
    x1 = -4. / po * d1 * r2 - 4. / pc * d3 * r4
    x2 = -4. / po * d1 * r5 - 4. / pc * d5 * r4
    e1, e2 = np.nanmean(x1), np.nanmean(x2)
    v1, v2 = np.nanmean(x1 ** 2) - e1 ** 2, np.nanmean(x2 ** 2) - e2 ** 2
    vt = v1 + v2
    s2 = (v2 * e1 + v1 * e2) / vt if vt > 0 else (e1 + e2) / 2.
    s = np.sqrt(np.abs(s2))
    return float(s * np.sign(s2)) if sign else float(s)


def amihud(r, dv, scale=1e6):
    """Ilichiditatea Amihud: media |r| (puncte de baza) la 1 milion USD tranzactionat."""
    x = pd.concat([r.rename('r'), dv.rename('dv')], axis=1, join='inner').dropna()
    x = x[x['dv'] > 0]
    return float((1e4 * x['r'].abs() / (x['dv'] / scale)).mean())


def daily_estimates(d, overnight=True):
    """Estimatorii de spread pe o fereastra de bare: overnight=True pentru date zilnice (ajustarea Corwin-Schultz
    pentru miscarea de peste noapte), overnight=False pentru barele intraday ale unei sesiuni."""
    rs, cov = roll_spread(d['close'])
    cs = cs_spread(d['high'], d['low'], d['close'] if overnight else None)
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    return pd.Series(dict(cov=cov, roll=rs, cs=cs.mean(), ar2=ar2.mean()))


def roll_mc(c, sigma, T=78, R=20000, seed=42):
    """Harris (1990): simulam modelul Roll (m_t mers aleator cu pasi N(0, sigma^2), q_t = +-1 cu prob. 1/2)
    pe T preturi si calculam covarianta de ordinul 1 a variatiilor (demediate), ca pe date reale."""
    rng = np.random.default_rng(seed)
    p = np.cumsum(rng.normal(0, sigma, (R, T)), axis=1) + c * np.where(rng.random((R, T)) < 0.5, 1, -1)
    dp = np.diff(p, axis=1)
    dm = dp - dp.mean(axis=1, keepdims=True)
    return (dm[:, 1:] * dm[:, :-1]).mean(axis=1)


def price_discovery(y, p=1):
    """VECM bivariat cu vectorul de cointegrare (1, -1) impus: dy_t = a0 + alpha (y1 - y2)_{t-1} + sum Gamma_i dy_{t-i} + e_t.
    Intoarce alpha, statisticile t, ponderea componentei Gonzalo-Granger (alpha_perp normalizat) si limitele
    ponderii informationale Hasbrouck (cele doua ordonari Cholesky)."""
    y = np.asarray(y, float)
    dy = np.diff(y, axis=0)
    z = (y[:, 0] - y[:, 1])[:-1]
    X = np.array([[1.0, z[t]] + [v for i in range(1, p + 1) for v in dy[t - i]] for t in range(p, len(dy))])
    Y = dy[p:]
    B = np.linalg.lstsq(X, Y, rcond=None)[0]
    U = Y - X @ B
    Om = U.T @ U / (len(U) - X.shape[1])
    alpha = B[1]
    se = np.sqrt(np.linalg.inv(X.T @ X)[1, 1] * np.diag(Om))
    ap = np.array([-alpha[1], alpha[0]])                       # alpha_perp: alpha' alpha_perp = 0
    cs = ap / ap.sum()
    psi = ap                                                    # randul comun al impactului pe termen lung (pana la o scala)
    IS = []
    for order in ([0, 1], [1, 0]):
        F = np.linalg.cholesky(Om[np.ix_(order, order)])
        v = (psi[order] @ F) ** 2 / (psi @ Om @ psi)
        IS.append(v[np.argsort(order)])
    IS = np.array(IS)
    return dict(alpha=alpha, t=alpha / se, cs=cs, is_lo=IS.min(axis=0), is_hi=IS.max(axis=0),
                corr=float(Om[0, 1] / np.sqrt(Om[0, 0] * Om[1, 1])), resid=U, n=len(U))


def pin_loglik(B, S, alpha, delta, mu, eb, es):
    """Log-verosimilitatea EKOP pentru o zi cu B cumparari si S vanzari, in forma factorizata care evita
    depasirea numerica (Lin si Ke, 2011): ln L = -eb - es + B ln(mu + eb) + S ln(mu + es) - ln B! - ln S!
    + ln[(1 - alpha) xb^B xs^S + alpha delta e^-mu xb^B + alpha (1 - delta) e^-mu xs^S], xb = eb / (mu + eb)."""
    from scipy.special import gammaln, logsumexp
    lxb, lxs = np.log(eb / (mu + eb)), np.log(es / (mu + es))
    terms = [np.log(1 - alpha) + B * lxb + S * lxs, np.log(alpha * delta) - mu + B * lxb,
             np.log(alpha * (1 - delta)) - mu + S * lxs]
    base = -eb - es + B * np.log(mu + eb) + S * np.log(mu + es) - gammaln(B + 1) - gammaln(S + 1)
    return float(base + logsumexp(terms)), terms


def walk_book(asks, qty):
    """Ordin de cumparare la piata de marime qty pe ofertele de vanzare asks = [(pret, cantitate), ...] crescator.
    Intoarce executiile, pretul mediu ponderat si ofertele ramase."""
    fills, left, rest = [], qty, []
    for p, q in asks:
        if left <= 0:
            rest.append((p, q))
            continue
        take = min(q, left)
        fills.append((p, take))
        left -= take
        if q > take:
            rest.append((p, q - take))
    vwap = sum(p * q for p, q in fills) / sum(q for _, q in fills)
    return fills, vwap, rest, left


def gm_quotes(theta, mu, vl, vh):
    """Cotatiile ask si bid: asteptarea valorii conditionat de o cumparare, respectiv de o vanzare.
    theta = P(V = vh), mu = ponderea investitorilor informati; cei neinformati cumpara sau vand cu prob. 1/2."""
    pb_h, pb_l = mu + (1 - mu) / 2, (1 - mu) / 2               # P(cumparare | V = vh), P(cumparare | V = vl)
    th_b = pb_h * theta / (pb_h * theta + pb_l * (1 - theta))   # P(V = vh | cumparare)
    th_s = pb_l * theta / (pb_l * theta + pb_h * (1 - theta))   # P(V = vh | vanzare)
    return vl + th_b * (vh - vl), vl + th_s * (vh - vl), th_b, th_s


def gm_simulate(mu, vl=90.0, vh=110.0, v=110.0, n=60, seed=1):
    """Un sir de tranzactii cand valoarea adevarata este v; formatorul de piata invata din fluxul de ordine."""
    rng = np.random.default_rng(seed)
    theta, rows = 0.5, []
    for t in range(n):
        ask, bid, th_b, th_s = gm_quotes(theta, mu, vl, vh)
        informed = rng.random() < mu
        buy = (v == vh) if informed else (rng.random() < 0.5)
        rows.append(dict(t=t, ask=ask, bid=bid, theta=theta, buy=buy))
        theta = th_b if buy else th_s
    return pd.DataFrame(rows)


def kyle(sigma_v, sigma_u):
    """Echilibrul Kyle: v ~ N(p0, sigma_v^2), cererea zgomotoasa u ~ N(0, sigma_u^2).
    Strategia informatului x = beta (v - p0), pretul p = p0 + lambda (x + u)."""
    beta = sigma_u / sigma_v
    lam = sigma_v / (2 * sigma_u)
    profit = sigma_v * sigma_u / 2
    return dict(beta=beta, lam=lam, profit=profit, post_var=sigma_v ** 2 / 2)


def ac_trajectory(X, T, N, sigma, eta, gamma, lam, eps=0.0):
    """Traiectoria optima de lichidare (formulele exacte in timp discret).
    X actiuni de vandut in T zile, N intervale; sigma = volatilitatea pretului (USD / zi^0.5);
    impact temporar h(v) = eps sgn + eta v; impact permanent g(v) = gamma v; lam = aversiunea la risc."""
    tau = T / N
    eta_t = eta - gamma * tau / 2
    t = np.arange(N + 1) * tau
    if lam == 0:
        x = X * (1 - t / T)
    else:
        kt2 = lam * sigma ** 2 / eta_t                       # kappa_tilde^2
        kappa = np.arccosh(kt2 * tau ** 2 / 2 + 1) / tau
        x = X * np.sinh(kappa * (T - t)) / np.sinh(kappa * T)
    n = -np.diff(x)
    E = 0.5 * gamma * X ** 2 + eps * X + eta_t / tau * np.sum(n ** 2)
    V = sigma ** 2 * tau * np.sum(x[1:] ** 2)
    return t, x, E, V


def ac_frontier(X, T, N, sigma, eta, gamma, lams):
    """Frontiera eficienta (abaterea standard, costul asteptat) pentru mai multe aversiuni la risc."""
    out = []
    for lam in lams:
        _, _, E, V = ac_trajectory(X, T, N, sigma, eta, gamma, lam)
        out.append((lam, E, np.sqrt(V)))
    return pd.DataFrame(out, columns=['lam', 'E', 'sd'])


def zi_simulate(n_events=300_000, L=40, rate_limit=1.0, rate_market=0.25, rate_cancel=0.025, n_ticks=4000, seed=7):
    """Ordine limita (uniform pe L tick-uri de la cotatia opusa), ordine la piata si anulari, toate de o unitate.
    Intoarce seria (mijloc, spread) si instantanee ale registrului."""
    rng = np.random.default_rng(seed)
    bid_q = np.zeros(n_ticks, int)
    ask_q = np.zeros(n_ticks, int)
    mid0 = n_ticks // 2
    bid_q[mid0 - 20:mid0] = 3
    ask_q[mid0 + 1:mid0 + 21] = 3
    rows, snaps, offset = [], [], 0
    for e in range(n_events):
        bb = np.flatnonzero(bid_q)[-1]
        ba = np.flatnonzero(ask_q)[0]
        n_orders = bid_q.sum() + ask_q.sum()
        w = np.array([2 * L * rate_limit, 2 * rate_market, rate_cancel * n_orders])
        u = rng.random() * w.sum()
        side = rng.random() < 0.5
        if u < w[0]:                                   # ordin limita
            k = rng.integers(1, L + 1)
            if side:
                p = ba - k
                bid_q[p] += 1
            else:
                p = bb + k
                ask_q[p] += 1
        elif u < w[0] + w[1]:                          # ordin la piata
            if side:
                ask_q[ba] -= 1 if ask_q[ba] > 0 else 0
            else:
                bid_q[bb] -= 1 if bid_q[bb] > 0 else 0
        else:                                          # anulare a unui ordin existent, ales uniform dintre toate ordinele
            side = rng.random() < bid_q.sum() / n_orders   # partea aleasa proportional cu numarul de ordine in asteptare
            if side and bid_q.sum() > 1:
                idx = rng.choice(np.flatnonzero(bid_q), p=bid_q[bid_q > 0] / bid_q.sum())
                bid_q[idx] -= 1
            elif (not side) and ask_q.sum() > 1:
                idx = rng.choice(np.flatnonzero(ask_q), p=ask_q[ask_q > 0] / ask_q.sum())
                ask_q[idx] -= 1
        if not bid_q.any():
            bid_q[bb] = 1
        if not ask_q.any():
            ask_q[ba] = 1
        bb = np.flatnonzero(bid_q)[-1]
        ba = np.flatnonzero(ask_q)[0]
        if bb < 5 * L or ba > n_ticks - 5 * L:         # recentrare: registrul ramane in mijlocul grilei
            sh = int((bb + ba) / 2 - n_ticks // 2)
            bid_q, ask_q = np.roll(bid_q, -sh), np.roll(ask_q, -sh)
            if sh > 0:
                bid_q[-sh:] = 0
                ask_q[-sh:] = 0
            else:
                bid_q[:-sh] = 0
                ask_q[:-sh] = 0
            offset += sh
        if e % 50 == 0:
            bb = np.flatnonzero(bid_q)[-1]
            ba = np.flatnonzero(ask_q)[0]
            rows.append((e, (bb + ba) / 2 - mid0 + offset, ba - bb))
        if e > n_events // 3 and e % 1000 == 0:
            bb = np.flatnonzero(bid_q)[-1]
            ba = np.flatnonzero(ask_q)[0]
            m = (bb + ba) / 2
            rel = np.arange(n_ticks) - m
            sel = np.abs(rel) <= 30
            snaps.append(pd.DataFrame({'rel': rel[sel], 'bid': bid_q[sel], 'ask': ask_q[sel]}))
    path = pd.DataFrame(rows, columns=['event', 'mid', 'spread'])
    return path, snaps, (bid_q, ask_q)

## Helper functions

In [ ]:
B = 1000

AC = {'X': 1000000.0, 'T': 5, 'N': 5, 'sigma': 0.95, 'eta': 2.5e-06, 'gamma': 2.5e-07}

C_MARKETS = {'BVB': {'ret': ('BETTR.INDX', 'close'), 'keys': ['TLV', 'SNP', 'H2O', 'BRD', 'SNG', 'SNN', 'TGN', 'EL', 'DIGI', 'TEL']}, 'US': {'ret': ('SPY.US', 'adjusted_close'), 'keys': ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GME', 'MSTR']}, 'Crypto': {'ret': ('BTC-USD.CC', 'close'), 'keys': ['ETH', 'SOL', 'XRP', 'DOGE', 'ADA', 'LTC']}}

PIN_PAR = {'alpha': 0.3, 'delta': 0.5, 'mu': 400.0, 'eb': 1000.0, 'es': 1000.0}

PIN_DAYS = [('A9: B = 1450, S = 1000', 1450, 1000), ('A10(a): B = 1000, S = 1420', 1000, 1420), ('A10(b): B = S = 1000', 1000, 1000), ('A10(d): B = 1195, S = 1000', 1195, 1000)]

ABO = {'col2': 0.4213, 'col6_v': 0.3354, 'col6_s': 0.0066, 'lo': 0.2, 'hi': 0.74, 'tax_lse': 0.419}

YEARS = (2021, 2025)

END_PD = '2026-09-18'

PD_START = '2024-09-19'

def spy_tod(s):
    """Profilul pe intervale de 5 minute: media |r| (pb), volumul median (milioane de actiuni), ponderea in volumul zilei."""
    x = s.dropna(subset=['r'])
    tod = x.groupby('tod').agg(absr=('r', lambda v: 1e4 * v.abs().mean()), vol=('volume', 'median'))
    share = (s['volume'] / s.groupby('date')['volume'].transform('sum')).groupby(s['tod']).mean()
    tod['share'] = 100 * share
    return tod


def intraday_spreads(s):
    """Estimatorii Roll, Corwin-Schultz si Abdi-Ranaldo calculati pe barele de 5 minute ale fiecarei zile."""
    rows = {}
    for d, g in s.dropna(subset=['close', 'high', 'low']).groupby('date'):
        rs, cov = roll_spread(g['close'])
        rows[d] = dict(cov=cov, cs=cs_spread(g['high'], g['low']).mean(),          # in cursul sesiunii: fara ajustare de noapte
                      
                       ar2=ar_terms(g['close'], g['high'], g['low']).mean(), price=g['close'].mean(),
                       edge=edge_spread(g['open'], g['high'], g['low'], g['close']),
                       var=np.var(np.diff(np.log(g['close'].values)), ddof=1))
    return pd.DataFrame(rows).T


def daily_spreads(key, start=START2):
    """Estimatorii din date zilnice pe ultimii doi ani (valori in puncte de baza)."""
    d = ohlc(key, start)
    rs, cov = roll_spread(d['close'])
    cs = cs_spread(d['high'], d['low'], d['close'])
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    return dict(roll=1e4 * rs if rs == rs else np.nan, roll_cov=float(cov), cs=1e4 * float(np.mean(cs)),
                edge=1e4 * edge_spread(d['open'], d['high'], d['low'], d['close']),
                ar=1e4 * float(np.sqrt(max(np.mean(ar2), 0))), ar_neg=bool(np.mean(ar2) <= 0), n=int(len(d)))


def spread_table():
    return {k: daily_spreads(k) for k in ASSETS}


def amihud_table(start=START2):
    out = {}
    for k in ASSETS:
        r = returns(k, start)
        dv = dollar_volume(k, start)
        out[k] = dict(illiq=amihud(r, dv), dv_med=float(dv.median() / 1e6), n=int(len(r)))
    return out


def spy_illiq_daily(s):
    """Ilichiditatea intrazilnica: media |r_5min| (pb) la 100 milioane USD tranzactionati, pe zi."""
    x = s.dropna(subset=['r', 'dv'])
    x = x[x['dv'] > 0]
    return (1e4 * x['r'].abs() / (x['dv'] / 1e8)).groupby(x['date']).mean().rename('illiq')


def sqrt_relation(s, nb=20):
    x = s.dropna(subset=['r', 'volume']).copy()
    x = x[x['volume'] > 0]
    x['part'] = x['volume'] / x.groupby('tod')['volume'].transform('median')
    x['z'] = x['r'].abs() / x.groupby('date')['r'].transform('std')
    bins = np.quantile(x['part'], np.linspace(0, 1, nb + 1))
    g = x.groupby(pd.cut(x['part'], bins, include_lowest=True)).agg(v=('part', 'mean'), z=('z', 'mean'))
    slope, icpt = np.polyfit(np.log(g['v']), np.log(g['z']), 1)
    return x, g, slope, icpt


def roll_small_sample(E):
    """Covarianta Roll pe zi (77 de randamente de 5 minute): ponderea zilelor cu covarianta pozitiva, observata si
    simulata sub modelul Roll adevarat (spread de un pas de cotare; fara spread), si deplasarea din demediere."""
    n = 77
    v = E['var'].mean()
    c_tick = (0.01 / E['price']).mean() / 2
    out = dict(pos_obs=float((E['cov'] > 0).mean()), n_days=int(len(E)), sd_bar=float(1e4 * np.sqrt(v)),
               c_tick=float(1e4 * c_tick))
    for tag, c in [('tick', c_tick), ('zero', 0.0)]:
        sig = np.sqrt(v - 2 * c ** 2)
        cov = np.array([roll_mc(c, sig, T=n + 1, R=len(E), seed=SEED + b) for b in range(200)])
        pos = (cov > 0).mean(axis=1)                        # 200 de "esantioane" de lungimea selectiei reale
        pooled = 1e4 * 2 * np.sqrt(np.clip(-cov.mean(axis=1), 0, None))
        out[f'pos_{tag}'] = float(pos.mean())
        out[f'pos_{tag}_lo'], out[f'pos_{tag}_hi'] = (float(x) for x in np.percentile(pos, [2.5, 97.5]))
        out[f'roll_{tag}'] = float(pooled.mean())
        out[f'roll_{tag}_lo'], out[f'roll_{tag}_hi'] = (float(x) for x in np.percentile(pooled, [2.5, 97.5]))
    # corectia deplasarii: E[cov_hat] ~ g1 - (g0 + 2 g1) / n  =>  g1 ~ (cov_mediu + var_medie / n) / (1 - 2 / n)
    g1 = (E['cov'].mean() + v / n) / (1 - 2 / n)
    out['roll_corr'] = float(1e4 * 2 * np.sqrt(max(-g1, 0)))
    out['roll5'] = float(1e4 * 2 * np.sqrt(max(-E['cov'].mean(), 0)))
    return out


def bet_etf_pair(start=PD_START):
    """Logaritmul preturilor de inchidere: ETF-ul Patria-TVBETETF (zilele cu tranzactii) si indicele BET-TR, zile comune."""
    e = read_market('TVBETETF.RO')
    e = e[e['volume'] > 0]
    idx = read_market('BETTR.INDX')
    j = pd.concat([e['close'].rename('etf'), idx['close'].rename('idx')], axis=1, join='inner').dropna().loc[start:END_PD]
    return np.log(j)


def race_daily(s):
    """V_t (USD) si sigma_t (%, anualizata) pentru fiecare zi completa; taxa zilnica implicata dupa col. 6 la V_t = Vbar (pb)."""
    s = s.loc[s['date'].dt.year.between(*YEARS)]
    d = pd.DataFrame({'V': s.groupby('date')['dv'].sum(),
                      'sigma': 100 * np.sqrt(252 * s.groupby('date')['r'].apply(lambda v: (v ** 2).sum()))})
    d['tax6'] = ABO['col6_v'] + ABO['col6_s'] * d['sigma']
    d['year'] = d.index.year
    return d


def race_prize(d):
    """Premiul anual implicat (USD milioane) dupa col. 2, col. 6 si scenariile extreme ale tabelului XIV."""
    out = {}
    for y, g in d.groupby('year'):
        V, Vbar = g['V'].sum(), g['V'].mean()
        out[int(y)] = dict(n=int(len(g)), V_tot=float(V), V_mean=float(Vbar), sigma_mean=float(g['sigma'].mean()),
                           col2=float(1e-4 * ABO['col2'] * V / 1e6),
                           col6=float(1e-4 * (ABO['col6_v'] * V + ABO['col6_s'] * Vbar * g['sigma'].sum()) / 1e6),
                           lo=float(1e-4 * ABO['lo'] * V / 1e6), hi=float(1e-4 * ABO['hi'] * V / 1e6))
    return out

## Analysis

In [ ]:
def boot_days(values_by_day, stat, B=B, seed=SEED, draws=False):
    """Bootstrap pe zile (blocuri = zile intregi): interval percentil de 95% pentru stat(esantion de zile).
    draws=True intoarce si valorile celor B reesantionari."""
    rng = np.random.default_rng(seed)
    days = np.array(list(values_by_day.keys()))
    out = []
    for _ in range(B):
        pick = rng.choice(days, len(days), replace=True)
        out.append(stat([values_by_day[d] for d in pick]))
    ci = (float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))
    return (ci + (np.array(out),)) if draws else ci


def a1_roll(var=0.0520, cov=-0.0081, price=20.0, rho_q=0.3):
    """Modelul Roll: momentele, estimatorul si deplasarea cand semnele tranzactiilor sunt autocorelate.
    Cu Corr(q_t, q_{t-1}) = rho_q (lant Markov, m independent de q): Cov(dp_t, dp_{t-1}) = -c^2 (1 - rho_q)^2."""
    c = np.sqrt(-cov)
    cov_rho = -c ** 2 * (1 - rho_q) ** 2
    return dict(c=c, s=2 * c, s_pct=100 * 2 * c / price, sig2=var - 2 * c ** 2, share=2 * c ** 2 / var,
                rho=cov / var, var=var, cov=cov, price=price, rho_q=rho_q, cov_rho=cov_rho,
                c_roll_rho=np.sqrt(-cov_rho), bias_pct=100 * rho_q)


def a2_roll_se(var=0.0520, cov=-0.0081, Ts=(78, 250)):
    """Eroarea standard a lui s_hat = 2 sqrt(-cov_hat) prin metoda delta; Var(cov_hat) din formula lui Bartlett
    pentru un MA(1) (aproximare gaussiana): T Var(cov_hat) -> g0^2 + 3 g1^2."""
    from scipy.stats import norm
    c = np.sqrt(-cov)
    out = dict(var=var, cov=cov, s=2 * c, avar=var ** 2 + 3 * cov ** 2)
    for T in Ts:
        se_g = np.sqrt((var ** 2 + 3 * cov ** 2) / T)
        se_s = se_g / c                               # ds/dg = -1 / sqrt(-g) = -1 / c
        out[T] = dict(se_g=se_g, se_s=se_s, cv=se_s / (2 * c), p_pos=norm.cdf(cov / se_g))
    return out


def gm_spread_formula(theta, mu, dv=20.0):
    """Spread-ul Glosten-Milgrom pentru theta general: a - b = 4 mu theta (1 - theta) dV / (1 - mu^2 (2 theta - 1)^2)."""
    return 4 * mu * theta * (1 - theta) * dv / (1 - mu ** 2 * (2 * theta - 1) ** 2)


def a3_gm(mu=0.3, theta=0.5, vl=90.0, vh=110.0):
    ask, bid, tb, ts = gm_quotes(theta, mu, vl, vh)
    ask2, bid2, tb2, ts2 = gm_quotes(tb, mu, vl, vh)          # dupa o cumparare
    a7, b7, _, _ = gm_quotes(0.7, mu, vl, vh)
    return dict(ask=ask, bid=bid, spread=ask - bid, pb_h=mu + (1 - mu) / 2, pb_l=(1 - mu) / 2, th_b=tb, th_s=ts,
                ask2=ask2, bid2=bid2, spread2=ask2 - bid2, th_b2=tb2, spread_07=a7 - b7,
                formula_07=gm_spread_formula(0.7, mu, vh - vl), formula_05=gm_spread_formula(0.5, mu, vh - vl))


def a4_gm_learning(mus=(0.1, 0.3, 0.5), level=108.0, vl=90.0, vh=110.0):
    """Dupa k cumparari consecutive, cota P(V_H) se inmulteste cu (1 + mu) / (1 - mu) la fiecare cumparare;
    bid-ul dupa k cumparari este V_L + dV theta_{k-1}, deci bid > level cere (k - 1) ln((1 + mu)/(1 - mu)) > ln(o*)."""
    q = (level - vl) / (vh - vl)
    out = {}
    for mu in mus:
        k = 1 + int(np.floor(np.log(q / (1 - q)) / np.log((1 + mu) / (1 - mu)))) + 1
        theta, kk = 0.5, 0
        while True:                                         # verificare directa cu cotatiile
            kk += 1
            ask, bid, tb, ts = gm_quotes(theta, mu, vl, vh)
            if bid > level:
                break
            theta = tb
        out[mu] = dict(k=kk - 1, k_formula=k, speed=np.log((1 + mu) / (1 - mu)))   # kk - 1 cumparari observate
    return out


def a5_kyle(sigma_v=2.0, sigma_u=10_000.0, y=15_000.0, sigma_u2=20_000.0):
    k = kyle(sigma_v, sigma_u)
    k2 = kyle(sigma_v, sigma_u2)
    k.update(sigma_v=sigma_v, sigma_u=sigma_u, y=y, dp=k['lam'] * y, lam100=100 * k['lam'],
             lam2=k2['lam'], profit2=k2['profit'], beta2=k2['beta'])
    return k


def a6_kyle_ols(sigma_v=2.0, sigma_u=10_000.0, p0=50.0, n=200_000, seed=SEED):
    """Economia Kyle simulata: panta OLS a lui (p - p0) pe y este lambda; un raport de tip Amihud |r| / (p0 |y|)
    estimeaza lambda / p0^2, iar cu volumul total |x| + |u| in loc de |y| il subestimeaza."""
    rng = np.random.default_rng(seed)
    k = kyle(sigma_v, sigma_u)
    v = p0 + sigma_v * rng.standard_normal(n)
    x = k['beta'] * (v - p0)
    u = sigma_u * rng.standard_normal(n)
    y = x + u
    p = p0 + k['lam'] * y
    slope = np.polyfit(y, p - p0, 1)[0]
    slope_v = np.polyfit(y, v - p0, 1)[0]
    r = np.abs(p - p0) / p0
    am_net = np.mean(r / (p0 * np.abs(y)))
    am_vol = np.mean(r / (p0 * (np.abs(x) + np.abs(u))))
    return dict(lam=k['lam'], slope=slope, slope_v=slope_v, target=k['lam'] / p0 ** 2, am_net=am_net, am_vol=am_vol,
                am_ratio=am_vol / am_net, p0=p0, var_y=np.var(y), var_y_theory=2 * sigma_u ** 2)


def a7_ac(lam=2e-6):
    """Almgren-Chriss: solutia in timp continuu (Euler-Lagrange: x'' = kappa^2 x) si programul discret exact."""
    tau = AC['T'] / AC['N']
    eta_t = AC['eta'] - AC['gamma'] * tau / 2
    kt2 = lam * AC['sigma'] ** 2 / eta_t
    kappa = np.arccosh(kt2 * tau ** 2 / 2 + 1) / tau
    kc = np.sqrt(lam * AC['sigma'] ** 2 / AC['eta'])          # timp continuu: eta_tilde -> eta cand tau -> 0
    t, x, E, V = ac_trajectory(lam=lam, **AC)
    t0, x0, E0, V0 = ac_trajectory(lam=0, **AC)
    xc = AC['X'] * np.sinh(kc * (AC['T'] - t)) / np.sinh(kc * AC['T'])
    return dict(lam=lam, eta_t=eta_t, kt2=kt2, kappa=kappa, kc=kc, kc2=kc ** 2, x=list(x), xc=list(xc),
                n=list(-np.diff(x)), E=E, sd=np.sqrt(V), E0=E0, sd0=np.sqrt(V0), sinh5=np.sinh(5 * kappa))


def a8_ac():
    return a7_ac(lam=2e-5)


def pin_posterior(B, S, alpha, delta, mu, eb, es):
    """Probabilitatile a posteriori ale celor trei stari (fara stire, stire proasta, stire buna) dupa (B, S)."""
    from scipy.special import softmax
    _, terms = pin_loglik(B, S, alpha, delta, mu, eb, es)
    return softmax(terms)


def a9_pin(B=1450, S=1000, **par):
    """Verosimilitatea EKOP pentru o zi: evaluarea directa esueaza numeric, forma factorizata nu."""
    import warnings as _w
    par = par or PIN_PAR
    a, d, mu, eb, es = (par[k] for k in ('alpha', 'delta', 'mu', 'eb', 'es'))
    pin = a * mu / (a * mu + eb + es)
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        naive = np.exp(-eb) * np.float64(eb) ** B                 # e^-1000 = 0 si 1000^1450 = inf: 0 * inf = nan
    ll, _ = pin_loglik(B, S, a, d, mu, eb, es)
    post = pin_posterior(B, S, a, d, mu, eb, es)
    return dict(pin=pin, naive=str(naive), exp_small=float(np.exp(-eb)), loglik=ll, p_none=post[0], p_bad=post[1],
                p_good=post[2], B=B, S=S, **par)


def a10_pin():
    """Aceiasi parametri: o zi cu stire proasta probabila (B = 1000, S = 1420), o zi linistita (B = S = 1000)
    si PIN cand mu se dubleaza."""
    x = a9_pin(B=1000, S=1420)
    q = a9_pin(B=1000, S=1000)
    par = dict(PIN_PAR, mu=800.0)
    pin2 = par['alpha'] * par['mu'] / (par['alpha'] * par['mu'] + par['eb'] + par['es'])
    return dict(bad=dict(p_none=x['p_none'], p_bad=x['p_bad'], p_good=x['p_good'], loglik=x['loglik']),
                quiet=dict(p_none=q['p_none'], p_bad=q['p_bad'], p_good=q['p_good']), pin=x['pin'], pin2=pin2)


def b1_ushape():
    s = intraday_spy()
    x = s.dropna(subset=['r'])
    by_day = {}
    for d, g in x.groupby('date'):
        g = g.set_index('tod')
        by_day[d] = (abs(g['r'].get('09:30', np.nan)), g.loc['11:30':'14:00', 'r'].abs().mean(),
                     g['volume'].iloc[-6:].sum() / g['volume'].sum())
    arr = np.array(list(by_day.values()))
    ratio = np.nanmean(arr[:, 0]) / np.nanmean(arr[:, 1])
    lo, hi = boot_days(by_day, lambda v: np.nanmean([a[0] for a in v]) / np.nanmean([a[1] for a in v]))
    vs = np.nanmean(arr[:, 2])
    vlo, vhi = boot_days(by_day, lambda v: np.nanmean([a[2] for a in v]))
    tod = spy_tod(s)
    # un pas al bootstrap-ului, explicit: prima reesantionare (acelasi generator ca boot_days)
    rng1 = np.random.default_rng(SEED)
    days = np.array(list(by_day.keys()))
    pick = rng1.choice(days, len(days), replace=True)
    a1 = np.array([by_day[d] for d in pick])
    first = dict(ratio=float(np.nanmean(a1[:, 0]) / np.nanmean(a1[:, 1])), n_unique=int(len(np.unique(pick))),
                 open=float(1e4 * np.nanmean(a1[:, 0])), mid=float(1e4 * np.nanmean(a1[:, 1])),
                 vshare=float(100 * np.nanmean(a1[:, 2])))
    # grafic: media |r| si ponderea in volumul zilei pe intervale, cu benzi bootstrap pe zile (benzi punctuale)
    piv = x.pivot_table(index='date', columns='tod', values='r', aggfunc=lambda v: abs(v).mean())
    vsh = (s['volume'] / s.groupby('date')['volume'].transform('sum')).groupby([s['date'], s['tod']]).sum().unstack()
    rng = np.random.default_rng(SEED)
    bs = np.array([piv.iloc[rng.integers(0, len(piv), len(piv))].mean().values for _ in range(300)])
    lo_b, hi_b = 1e4 * np.nanpercentile(bs, 2.5, axis=0), 1e4 * np.nanpercentile(bs, 97.5, axis=0)
    rng = np.random.default_rng(SEED + 1)
    bv = np.array([vsh.iloc[rng.integers(0, len(vsh), len(vsh))].mean().values for _ in range(300)])
    lo_v, hi_v = 100 * np.percentile(bv, 2.5, axis=0), 100 * np.percentile(bv, 97.5, axis=0)
    k = np.arange(len(piv.columns))
    cols = list(piv.columns)
    i0, i1 = cols.index('11:30'), cols.index('14:00')
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    ax = axes[0]
    ax.axvspan(i0 - 0.5, i1 + 0.5, color=Teal, alpha=0.12, lw=0, label='Midday window: bars starting 11:30-14:00')
    ax.fill_between(k, lo_b, hi_b, color='#DADADA', alpha=0.8, lw=0, label='Pointwise 95% bootstrap band (resampling days)')
    ax.plot(k, 1e4 * piv.mean().values, color=IDAred, lw=1.2, marker='o', ms=2.5, label='Mean |5-minute return| (bp)')
    ax.plot(0, 1e4 * piv.mean().values[0], 'o', color=MainBlue, ms=7, label='First bar (09:30)')
    ax.set_xticks(k[::12])
    ax.set_xticklabels(cols[::12], fontsize=7.5)
    ax.set_xlabel('Start of the 5-minute bar (New York time)')
    ax.set_ylabel('Mean |return| (bp)')
    ax.set_title('Volatility: high at the open, low at midday', fontsize=9, loc='left')
    ax = axes[1]
    ax.fill_between(k, lo_v, hi_v, color='#DADADA', alpha=0.8, lw=0)
    ax.plot(k, 100 * vsh.mean().values, color=MainBlue, lw=1.2, marker='o', ms=2.5, label='Mean share of daily volume (%)')
    ax.plot(k[-6:], 100 * vsh.mean().values[-6:], 'o', color=Amber, ms=5, label='Last six bars (15:30-15:55)')
    ax.axhline(100 / 78, color=Gray, lw=0.8, ls='--', label='Even volume, 1/78 = 1.28%')
    ax.set_xticks(k[::12])
    ax.set_xticklabels(cols[::12], fontsize=7.5)
    ax.set_xlabel('Start of the 5-minute bar (New York time)')
    ax.set_ylabel('Share of daily volume (%)')
    ax.set_title('Volume: U-shaped, heaviest at the close', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.13, 1, 1))
    save_fig('ch10_sem_ushape')
    return dict(ratio=ratio, lo=lo, hi=hi, open=1e4 * np.nanmean(arr[:, 0]), mid=1e4 * np.nanmean(arr[:, 1]),
                vshare=100 * vs, vlo=100 * vlo, vhi=100 * vhi, n=len(by_day), absr_close=float(tod['absr'].iloc[-1]),
                n_mid_bars=int(i1 - i0 + 1), first=first)


def b2_spreads():
    s = intraday_spy()
    E = intraday_spreads(s)
    by_day = {d: row for d, row in E.iterrows()}
    roll5 = 1e4 * 2 * np.sqrt(max(-E['cov'].mean(), 0))
    rlo, rhi = boot_days(by_day, lambda v: 1e4 * 2 * np.sqrt(max(-np.mean([r['cov'] for r in v]), 0)))
    cs5 = 1e4 * E['cs'].mean()
    clo, chi = boot_days(by_day, lambda v: 1e4 * np.mean([r['cs'] for r in v]))
    ar5 = 1e4 * np.sqrt(max(E['ar2'].mean(), 0))
    alo, ahi = boot_days(by_day, lambda v: 1e4 * np.sqrt(max(np.mean([r['ar2'] for r in v]), 0)))
    tick = 1e4 * (0.01 / E['price']).mean()
    # date zilnice: bootstrap pe blocuri mobile de 20 de zile
    d = ohlc('SPY', START2)
    cs = cs_spread(d['high'], d['low'], d['close'])
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    dp = np.diff(np.log(d['close'].values))
    pairs = np.column_stack([dp[1:], dp[:-1]])                # perechile (dp_t, dp_{t-1}) pentru covarianta Roll
    roll_cov = lambda x: np.mean(x[:, 0] * x[:, 1]) - x[:, 0].mean() * x[:, 1].mean()
    rng = np.random.default_rng(SEED)
    n, L = len(cs), 20
    csb, arb, rlb = [], [], []
    for _ in range(B):
        st = rng.integers(0, n - L + 1, n // L + 1)             # inceputurile posibile ale blocurilor: 0, ..., n - L
        idx = np.concatenate([np.arange(a, a + L) for a in st])[:n]
        csb.append(1e4 * cs[idx].mean())
        arb.append(1e4 * np.sqrt(max(ar2[idx].mean(), 0)))
        ip = np.minimum(idx, len(pairs) - 1)
        rlb.append(roll_cov(pairs[ip]))
    rlb = np.array(rlb)
    roll_d = 1e4 * 2 * np.sqrt(max(-roll_cov(pairs), 0))
    rlb_s = 1e4 * 2 * np.sqrt(np.maximum(-rlb, 0))
    sd_daily = 1e4 * np.log(d['close']).diff().std()
    res = dict(roll5=roll5, rlo=rlo, rhi=rhi, cs5=cs5, clo=clo, chi=chi, ar5=ar5, alo=alo, ahi=ahi, tick=tick,
               cs_d=1e4 * cs.mean(), cs_dlo=float(np.percentile(csb, 2.5)), cs_dhi=float(np.percentile(csb, 97.5)),
               ar_d=1e4 * np.sqrt(max(ar2.mean(), 0)), ar_dlo=float(np.percentile(arb, 2.5)),
               ar_dhi=float(np.percentile(arb, 97.5)), sd_daily=sd_daily, share_negcov=float((E['cov'] < 0).mean()),
               roll_d=float(roll_d), roll_dlo=float(np.percentile(rlb_s, 2.5)), roll_dhi=float(np.percentile(rlb_s, 97.5)),
               roll_d_pos=float((rlb >= 0).mean()))
    # cat de des este trunchiat la 0 fiecare estimator in reesantionari (momentul mediat are semnul gresit)
    _, _, mc = boot_days(by_day, lambda v: np.mean([r['cov'] for r in v]), draws=True)
    _, _, ma = boot_days(by_day, lambda v: np.mean([r['ar2'] for r in v]), draws=True)
    res.update(trunc_roll5=float((mc >= 0).mean()), trunc_ar5=float((ma <= 0).mean()),
               trunc_ar_d=float((np.array(arb) == 0).mean()), trunc_roll_d=float((rlb >= 0).mean()),
               n_days5=int(len(E)), n_daily=int(len(d)), n_ret5=int(s.dropna(subset=['r']).groupby('date').size().iloc[0] - 1))
    # aceeasi fereastra de doi ani pentru barele de 5 minute (sensibilitate: frecventa fara efectul perioadei)
    Em = E.loc[START2:]
    bm = {dd: row for dd, row in Em.iterrows()}
    res.update(n_days5m=int(len(Em)),
               roll5m=float(1e4 * 2 * np.sqrt(max(-Em['cov'].mean(), 0))),
               cs5m=float(1e4 * Em['cs'].mean()), ar5m=float(1e4 * np.sqrt(max(Em['ar2'].mean(), 0))))
    res['roll5mlo'], res['roll5mhi'] = boot_days(bm, lambda v: 1e4 * 2 * np.sqrt(max(-np.mean([r['cov'] for r in v]), 0)))
    res['cs5mlo'], res['cs5mhi'] = boot_days(bm, lambda v: 1e4 * np.mean([r['cs'] for r in v]))
    res['ar5mlo'], res['ar5mhi'] = boot_days(bm, lambda v: 1e4 * np.sqrt(max(np.mean([r['ar2'] for r in v]), 0)))
    fig_b2_estimates(res)
    return res


def b3_cross():
    T = spread_table()
    A = amihud_table()
    rows = []
    for k in ASSETS:
        r = returns(k, START2)
        ppy = 365 if ASSETS[k][2] == 'Crypto' else 252
        rows.append(dict(key=k, grp=ASSETS[k][2], cs=T[k]['cs'], ar=T[k]['ar'], vol=100 * r.std() * np.sqrt(ppy),
                         illiq=A[k]['illiq'], dv=A[k]['dv_med']))
    t = pd.DataFrame(rows).set_index('key')
    rho_cs_vol = stats.spearmanr(t['cs'], t['vol'])
    rho_il_vol = stats.spearmanr(t['illiq'], t['vol'])
    rho_cs_il = stats.spearmanr(t['cs'], t['illiq'])
    rho_il_dv = stats.spearmanr(t['illiq'], t['dv'])
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
    show = {'SPY', 'GME', 'MSTR', 'TLV', 'H2O', 'TEL', 'TVBETETF', 'BTC', 'DOGE'}
    for g, c in GROUP_COL.items():
        m = t['grp'] == g
        axes[0].scatter(t.loc[m, 'vol'], t.loc[m, 'cs'], color=c, s=26, label=g)
        axes[1].scatter(t.loc[m, 'dv'], t.loc[m, 'illiq'], color=c, s=26)
        axes[2].scatter(t.loc[m, 'illiq'], t.loc[m, 'cs'], color=c, s=26)
    for k in show:
        axes[0].annotate(LABELS[k], (t.loc[k, 'vol'], t.loc[k, 'cs']), fontsize=6.5, color='black',
                         xytext=(3, 2), textcoords='offset points')
        axes[1].annotate(LABELS[k], (t.loc[k, 'dv'], t.loc[k, 'illiq']), fontsize=6.5, color='black',
                         xytext=(3, 2), textcoords='offset points')
        axes[2].annotate(LABELS[k], (t.loc[k, 'illiq'], t.loc[k, 'cs']), fontsize=6.5, color='black',
                         xytext=(3, 2), textcoords='offset points')
    axes[0].set_xlabel('Annualised volatility (%)')
    axes[0].set_ylabel('Corwin-Schultz spread (bp)')
    axes[0].set_title(f'(1) Spearman {rho_cs_vol.statistic:.2f}: tracks volatility', fontsize=9, loc='left')
    for ax in axes[1:]:
        ax.set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Median daily traded value (USD million)')
    axes[1].set_ylabel('Amihud illiquidity (bp per USD 1m)')
    axes[1].set_title(f'(2) Spearman {rho_il_dv.statistic:.2f}: tracks traded value', fontsize=9, loc='left')
    axes[2].set_xlabel('Amihud illiquidity (bp per USD 1m)')
    axes[2].set_ylabel('Corwin-Schultz spread (bp)')
    axes[2].set_title(f'(3) Spearman {rho_cs_il.statistic:.2f}: the two disagree', fontsize=9, loc='left')
    h, l = axes[0].get_legend_handles_labels()
    fig_legend_bottom(fig, h, [x + ' (daily bars, Sep 2024 - Sep 2026)' for x in l], ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_cross')
    grp_stats = {g: dict(cs_med=float(t.loc[t['grp'] == g, 'cs'].median()), ar_med=float(t.loc[t['grp'] == g, 'ar'].median()),
                         ar_zero=int((t.loc[t['grp'] == g, 'ar'] == 0).sum()), n=int((t['grp'] == g).sum()))
                 for g in ('US', 'BVB', 'Crypto')}
    return dict(rho_cs_vol=float(rho_cs_vol.statistic), p_cs_vol=float(rho_cs_vol.pvalue),
                rho_il_vol=float(rho_il_vol.statistic), rho_cs_il=float(rho_cs_il.statistic),
                rho_il_dv=float(rho_il_dv.statistic), p_il_dv=float(rho_il_dv.pvalue),
                n=int(len(t)), n_ar_zero=int((t['ar'] == 0).sum()),
                bvb_cs_med=float(t.loc[t['grp'] == 'BVB', 'cs'].median()), us_cs_med=float(t.loc[t['grp'] == 'US', 'cs'].median()),
                cr_cs_med=float(t.loc[t['grp'] == 'Crypto', 'cs'].median()), groups=grp_stats,
                p_cs_il=float(rho_cs_il.pvalue), table=t.round(4).to_dict(orient='index'))


def b4_illiq_vix():
    s = intraday_spy()
    ill = spy_illiq_daily(s)
    vix = read_market('VIX.INDX')['close'].rename('vix')
    j = pd.concat([ill, vix], axis=1, join='inner').dropna()
    y, X = np.log(j['illiq']), sm.add_constant(np.log(j['vix']))
    ols = sm.OLS(y, X).fit()
    hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    # control pentru nivelul valorii tranzactionate (tendinta)
    dvd = s.groupby('date')['dv'].sum().rename('dv')
    j2 = pd.concat([j, dvd], axis=1, join='inner').dropna()
    X2 = sm.add_constant(pd.DataFrame({'lvix': np.log(j2['vix']), 't': np.arange(len(j2)) / 252}))
    hac2 = sm.OLS(np.log(j2['illiq']), X2).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    rho1 = float(np.corrcoef(ols.resid[1:], ols.resid[:-1])[0, 1])
    # (d) stabilitatea elasticitatii: ruptura la mijlocul esantionului (data fixata inainte de estimare), test Wald HAC
    mid = j.index[len(j) // 2]
    post = (j.index >= mid).astype(float)
    X3 = sm.add_constant(pd.DataFrame({'lvix': np.log(j['vix']), 'post': post, 'lvix_post': post * np.log(j['vix'])},
                                      index=j.index))
    hac3 = sm.OLS(y, X3).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    w = hac3.wald_test('lvix_post = 0', scalar=True)
    cv3 = hac3.cov_params()
    se_post = float(np.sqrt(cv3.loc['lvix', 'lvix'] + cv3.loc['lvix_post', 'lvix_post'] + 2 * cv3.loc['lvix', 'lvix_post']))
    fig_b4(j, ols, hac, hac2, hac3, se_post, mid)
    return dict(se_post=se_post, wald=float(w.statistic), se_pre=float(hac3.bse['lvix']), a=float(hac.params.iloc[0]),
                illiq_mean=float(j['illiq'].mean()), vix_mean=float(j['vix'].mean()),
                start=j.index[0].strftime('%Y-%m-%d'), end=j.index[-1].strftime('%Y-%m-%d'), b=float(hac.params.iloc[1]), se_ols=float(ols.bse.iloc[1]), se_hac=float(hac.bse.iloc[1]),
                brk=mid.strftime('%Y-%m-%d'), b_pre=float(hac3.params['lvix']),
                b_post=float(hac3.params['lvix'] + hac3.params['lvix_post']), d_brk=float(hac3.params['lvix_post']),
                se_brk=float(hac3.bse['lvix_post']), p_brk=float(w.pvalue),
                lo=float(hac.params.iloc[1] - 1.96 * hac.bse.iloc[1]), hi=float(hac.params.iloc[1] + 1.96 * hac.bse.iloc[1]),
                r2=float(ols.rsquared), n=int(len(j)), rho1=rho1, b2=float(hac2.params['lvix']),
                se2=float(hac2.bse['lvix']), trend=float(hac2.params['t']), se_trend=float(hac2.bse['t']))


def b5_amihud_groups():
    per = {}
    for a, b in [('2024-09-19', '2025-09-18'), ('2025-09-19', '2026-09-18')]:
        per[a[:4]] = {k: float((1e4 * returns(k, a, b).abs() / (dollar_volume(k, a, b) / 1e6)).dropna().mean())
                      for k in ASSETS}
    p1, p2 = pd.Series(per['2024']), pd.Series(per['2025'])
    rho = stats.spearmanr(p1, p2)
    A = amihud_table()
    il = pd.Series({k: v['illiq'] for k, v in A.items()})
    grp = pd.Series({k: v[2] for k, v in ASSETS.items()})
    rng = np.random.default_rng(SEED)
    ratios = []
    for _ in range(B):
        bb = il[grp == 'BVB'].sample(frac=1, replace=True, random_state=rng.integers(1e9)).median()
        uu = il[grp == 'US'].sample(frac=1, replace=True, random_state=rng.integers(1e9)).median()
        ratios.append(bb / uu)
    fig_b5(il, grp, np.array(ratios), p1, p2, float(il[grp == 'BVB'].median() / il[grp == 'US'].median()))
    return dict(rho=float(rho.statistic), p=float(rho.pvalue), med_bvb=float(il[grp == 'BVB'].median()),
                med_us=float(il[grp == 'US'].median()), med_cr=float(il[grp == 'Crypto'].median()),
                ratio=float(il[grp == 'BVB'].median() / il[grp == 'US'].median()),
                lo=float(np.percentile(ratios, 2.5)), hi=float(np.percentile(ratios, 97.5)))


def b6_sqrt():
    s = intraday_spy()
    x, g, slope, _ = sqrt_relation(s)
    days = x['date'].unique()
    by = {d: grp for d, grp in x.groupby('date')}
    bins = np.quantile(x['part'], np.linspace(0, 1, 21))
    rng = np.random.default_rng(SEED)
    sl = []
    for _ in range(300):
        pick = rng.choice(days, len(days), replace=True)
        xx = pd.concat([by[d] for d in pick])
        gg = xx.groupby(pd.cut(xx['part'], bins, include_lowest=True)).agg(v=('part', 'mean'), z=('z', 'mean')).dropna()
        sl.append(np.polyfit(np.log(gg['v']), np.log(gg['z']), 1)[0])
    lo, hi = np.percentile(sl, [2.5, 97.5])
    # pe barele individuale (nu pe grupe)
    y = np.log(x['r'].abs() + 1e-6) - np.log(x.groupby('date')['r'].transform('std'))
    raw = sm.OLS(y, sm.add_constant(np.log(x['part']))).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(x['date'])[0]})
    # randamentele nule: ln|r| nu este definit; constanta 1e-6 (0,01 pb) si, ca sensibilitate, excluderea barelor cu r = 0
    nz = (x['r'] != 0).values
    raw_nz = sm.OLS(y[nz], sm.add_constant(np.log(x['part'][nz]))).fit(cov_type='cluster',
                                                                       cov_kwds={'groups': pd.factorize(x['date'][nz])[0]})
    # (c) variabila instrumentala: volumul relativ al aceleiasi bare (aceeasi ora) din ziua precedenta
    x = x.assign(ly=y, lv=np.log(x['part']))
    x['lv_lag'] = x.groupby('tod')['lv'].shift(1)             # barele sunt ordonate in timp; shift pe ora = ziua anterioara
    x['ly_lag'] = x.groupby('tod')['ly'].shift(1)
    z = x.dropna(subset=['lv_lag', 'ly_lag'])
    gid = pd.factorize(z['date'])[0]
    fs = sm.OLS(z['lv'], sm.add_constant(z['lv_lag'])).fit(cov_type='cluster', cov_kwds={'groups': gid})
    Z = sm.add_constant(z['lv_lag']).values
    Xe = sm.add_constant(z['lv']).values
    Pz = Z @ np.linalg.solve(Z.T @ Z, Z.T @ Xe)               # proiectia regresorilor pe instrumente
    b_iv = np.linalg.solve(Pz.T @ Xe, Pz.T @ z['ly'].values)
    e = z['ly'].values - Xe @ b_iv
    A = np.linalg.inv(Pz.T @ Pz)
    meat = sum(np.outer(Pz[gid == gg].T @ e[gid == gg], Pz[gid == gg].T @ e[gid == gg]) for gg in np.unique(gid))
    se_iv = np.sqrt(np.diag(A @ meat @ A))
    red = np.corrcoef(z['ly'], z['ly_lag'])[0, 1]            # |r| de ieri la aceeasi ora: canalul care incalca excluderea
    fig_b6(g, slope, (lo, hi), raw, raw_nz, b_iv[1], se_iv[1])
    return dict(slope=float(slope), lo=float(lo), hi=float(hi), n=int(len(x)), n_days=int(len(days)),
                raw=float(raw.params.iloc[1]), raw_se=float(raw.bse.iloc[1]), zero_share=float(1 - nz.mean()),
                raw_nz=float(raw_nz.params.iloc[1]), raw_nz_se=float(raw_nz.bse.iloc[1]), fs=float(fs.params.iloc[1]),
                fs_t=float(fs.tvalues.iloc[1]), iv=float(b_iv[1]), iv_se=float(se_iv[1]), n_iv=int(len(z)),
                corr_absr_lag=float(red))


def b7_btc():
    b = intraday_btc().dropna(subset=['r'])
    wk = b.index.dayofweek >= 5
    h = b.index.hour
    by_day = {}
    for d, g in b.groupby('date'):
        hh = g.index.hour
        by_day[d] = (g['r'].abs()[(hh >= 13) & (hh < 17)].mean(), g['r'].abs()[(hh >= 3) & (hh < 11)].mean(),
                     g['r'].abs().mean(), d.dayofweek >= 5)
    wd = {d: v for d, v in by_day.items() if not v[3]}
    ratio = np.nanmean([v[0] for v in wd.values()]) / np.nanmean([v[1] for v in wd.values()])
    lo, hi = boot_days(wd, lambda v: np.nanmean([a[0] for a in v]) / np.nanmean([a[1] for a in v]))
    wkr = np.nanmean([v[2] for v in by_day.values() if not v[3]]) / np.nanmean([v[2] for v in by_day.values() if v[3]])
    lo2, hi2 = boot_days(by_day, lambda v: np.nanmean([a[2] for a in v if not a[3]]) / np.nanmean([a[2] for a in v if a[3]]))
    nb = b.groupby('date').size()                                     # barele cu randament valid pe zi (din 288)
    cover = dict(full=int((nb == 288).sum()), min=int(nb.min()), med=float(nb.median()),
                 n_wd=int(sum(1 for v in by_day.values() if not v[3])), n_we=int(sum(1 for v in by_day.values() if v[3])))
    fig_b7(b, (ratio, lo, hi), (wkr, lo2, hi2))
    return dict(ratio=float(ratio), lo=lo, hi=hi, wk=float(wkr), wklo=lo2, wkhi=hi2, n_days=len(by_day), cover=cover,
                us_d=float(1e4 * np.nanmean([v[0] for v in wd.values()])), asia_d=float(1e4 * np.nanmean([v[1] for v in wd.values()])),
                wd_d=float(1e4 * np.nanmean([v[2] for v in by_day.values() if not v[3]])),
                we_d=float(1e4 * np.nanmean([v[2] for v in by_day.values() if v[3]])),
                us=float(1e4 * b['r'].abs()[(~wk) & (h >= 13) & (h < 17)].mean()),
                asia=float(1e4 * b['r'].abs()[(~wk) & (h >= 3) & (h < 11)].mean()))


def fig_b2_estimates(r):
    """Estimari si intervale pe scala logaritmica; intervalele care ating 0 sunt taiate la marginea stanga."""
    rows = [('Roll', 'roll5', 'rlo', 'rhi', 'roll5m', 'roll5mlo', 'roll5mhi', 'roll_d', 'roll_dlo', 'roll_dhi'),
            ('Corwin-Schultz', 'cs5', 'clo', 'chi', 'cs5m', 'cs5mlo', 'cs5mhi', 'cs_d', 'cs_dlo', 'cs_dhi'),
            ('Abdi-Ranaldo', 'ar5', 'alo', 'ahi', 'ar5m', 'ar5mlo', 'ar5mhi', 'ar_d', 'ar_dlo', 'ar_dhi')]
    series = [('5-minute bars, Oct 2020 - Sep 2026 (day bootstrap)', MainBlue, 1, -0.22),
              ('5-minute bars, Sep 2024 - Sep 2026 (day bootstrap)', Teal, 4, 0.0),
              ('Daily bars, Sep 2024 - Sep 2026 (20-day block bootstrap)', IDAred, 7, 0.22)]
    left = 0.1
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for i, row in enumerate(rows):
        for lab, c, k, off in series:
            est, lo, hi = r[row[k]], r[row[k + 1]], r[row[k + 2]]
            y = i + off
            ax.hlines(y, max(lo, left), hi, color=c, lw=1.6)
            ax.plot(est, y, 'o', color=c, ms=6, label=lab if i == 0 else None)
            if lo <= left:
                ax.plot(left * 1.02, y, marker='<', color=c, ms=6)
            ax.text(hi * 1.12, y, f'{est:.2f}', va='center', fontsize=7.5, color='black')
    ax.axvline(r['tick'], color=Gray, ls='--', lw=0.9)
    ax.text(r['tick'] * 1.08, -0.42, f"one tick = {r['tick']:.2f} bp", fontsize=7.5, color='black')
    ax.set_xscale('log')
    ax.set_xlim(left, 400)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([x[0] for x in rows])
    ax.invert_yaxis()
    ax.set_xlabel('Relative spread estimate (bp, log scale); a triangle marks an interval that reaches 0')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    plt.tight_layout()
    save_fig('ch10_sem_b2_estimates')


def b2_mc(E):
    """Monte Carlo sub modelul Roll, calibrat pe SPY (aceleasi setari ca roll_small_sample): distributiile ponderii
    zilelor cu covarianta pozitiva si ale estimarii Roll agregate, pentru un spread de un pas si fara spread."""
    n = 77
    v = E['var'].mean()
    c_tick = (0.01 / E['price']).mean() / 2
    obs_pos = float((E['cov'] > 0).mean())
    obs_roll = float(1e4 * 2 * np.sqrt(max(-E['cov'].mean(), 0)))
    g1 = (E['cov'].mean() + v / n) / (1 - 2 / n)
    corr = float(1e4 * 2 * np.sqrt(max(-g1, 0)))
    sims = {}
    for tag, c in [('tick', c_tick), ('zero', 0.0)]:
        sig = np.sqrt(v - 2 * c ** 2)
        cov = np.array([roll_mc(c, sig, T=n + 1, R=len(E), seed=SEED + b) for b in range(200)])
        sims[tag] = dict(pos=(cov > 0).mean(axis=1), pooled=1e4 * 2 * np.sqrt(np.clip(-cov.mean(axis=1), 0, None)))
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
    for ax, key, xl, obs in [(axes[0], 'pos', 'Share of days with a positive covariance (%)', 100 * obs_pos),
                             (axes[1], 'pooled', 'Pooled Roll estimate (bp)', obs_roll)]:
        sc = 100 if key == 'pos' else 1
        for tag, c, lab in [('zero', MainBlue, 'Simulated, no spread (c = 0)'),
                            ('tick', Amber, f'Simulated, one-tick spread (c = {1e4 * c_tick:.2f} bp)')]:
            ax.hist(sc * sims[tag][key], bins=25, color=c, alpha=0.55, label=lab)
        ax.axvline(obs, color=IDAred, lw=1.6, label='Observed, SPY 5-minute bars')
        if key == 'pooled':
            ax.axvline(corr, color=Purple, lw=1.4, ls='--', label='Observed, bias-corrected')
        ax.set_xlabel(xl)
        ax.set_ylabel('Number of simulated samples (of 200)')
    axes[0].set_title('Positive covariances are common even under Roll', fontsize=9, loc='left')
    axes[1].set_title('Demeaning alone produces about 2 bp', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + [h2[-1]], l1 + [l2[-1]], ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    save_fig('ch10_sem_b2_mc')
    return dict(pos_tick=float(sims['tick']['pos'].mean()), pos_zero=float(sims['zero']['pos'].mean()),
                roll_tick=float(sims['tick']['pooled'].mean()), roll_zero=float(sims['zero']['pooled'].mean()),
                obs_pos=obs_pos, obs_roll=obs_roll, roll_corr=corr, var_bar=float(v), n=n, n_days=int(len(E)))


def fig_b4(j, ols, hac, hac2, hac3, se_post, mid):
    """Relatia log-log cu dreapta OLS, autocorelatia reziduurilor si intervalele HAC ale elasticitatii."""
    x, y = np.log(j['vix']), np.log(j['illiq'])
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[1.1, 1, 1.1]))
    ax = axes[0]
    ax.scatter(x, y, s=4, color=MainBlue, alpha=0.35, label='One trading day')
    xs = np.linspace(x.min(), x.max(), 20)
    ax.plot(xs, ols.params.iloc[0] + ols.params.iloc[1] * xs, color=IDAred, lw=1.6,
            label=f'OLS fit, slope {ols.params.iloc[1]:.2f}')
    ax.set_xlabel('log VIX')
    ax.set_ylabel('log intraday illiquidity')
    ax.set_title('(1) Elasticity close to 1', fontsize=9, loc='left')
    ax = axes[1]
    e = ols.resid.values
    lags = np.arange(1, 31)
    acf = [np.corrcoef(e[l:], e[:-l])[0, 1] for l in lags]
    ax.bar(lags, acf, color=Amber, width=0.7, label='Residual autocorrelation')
    b = 1.96 / np.sqrt(len(e))
    ax.axhspan(-b, b, color='#DADADA', alpha=0.8, lw=0)
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_xlabel('Lag (trading days)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title('(2) Residuals are persistent', fontsize=9, loc='left')
    ax = axes[2]
    rows = [('Baseline, OLS SE', ols.params.iloc[1], ols.bse.iloc[1], MainBlue),
            ('Baseline, HAC SE', hac.params.iloc[1], hac.bse.iloc[1], IDAred),
            ('With time trend, HAC', hac2.params['lvix'], hac2.bse['lvix'], Forest),
            (f'Before {mid:%d %b %Y}, HAC', hac3.params['lvix'], hac3.bse['lvix'], Purple),
            (f'From {mid:%d %b %Y}, HAC', hac3.params['lvix'] + hac3.params['lvix_post'], se_post, Teal)]
    for i, (lab, bb, se, c) in enumerate(rows):
        ax.hlines(i, bb - 1.96 * se, bb + 1.96 * se, color=c, lw=2)
        ax.plot(bb, i, 'o', color=c, ms=6)
        ax.text(bb + 1.96 * se + 0.03, i, f'{bb:.2f}', va='center', fontsize=7.5, color='black')
    ax.axvline(1, color=Gray, ls='--', lw=0.8)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.7)
    ax.set_xlabel('Elasticity with 95% interval')
    ax.set_title('(3) Specifications', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_b4')


def fig_b5(il, grp, ratios, p1, p2, ratio):
    """Amihud pe active (scala log), distributia bootstrap a raportului medianelor, clasamentele pe doi ani."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
    ax = axes[0]
    xpos = 0
    for g in ('US', 'Crypto', 'BVB'):
        v = il[grp == g].sort_values()
        xs = np.arange(xpos, xpos + len(v))
        ax.scatter(xs, v.values, color=GROUP_COL[g], s=24, label=g)
        ax.hlines(v.median(), xs[0] - 0.4, xs[-1] + 0.4, color=GROUP_COL[g], lw=1.4)
        xpos += len(v) + 1
    ax.set_yscale('log')
    ax.set_xticks([])
    ax.set_ylabel('Amihud ratio (bp per USD 1m, log)')
    ax.set_title('(1) Assets and group medians (lines)', fontsize=9, loc='left')
    ax = axes[1]
    ax.hist(np.log10(ratios), bins=30, color=Purple, alpha=0.7, label='Bootstrap draws of BVB median / US median')
    lo, hi = np.percentile(ratios, [2.5, 97.5])
    for v_, ls in [(lo, ':'), (hi, ':'), (ratio, '-')]:
        ax.axvline(np.log10(v_), color=IDAred, ls=ls, lw=1.3)
    ax.set_xlabel('log10 of the ratio (red: estimate and 95% interval)')
    ax.set_ylabel('Draws (of 1000)')
    ax.set_title('(2) Resampling assets within groups', fontsize=9, loc='left')
    ax = axes[2]
    for g in ('US', 'BVB', 'Crypto'):
        m = (grp == g)
        ax.scatter(p1[m.index[m]], p2[m.index[m]], color=GROUP_COL[g], s=22)
    lim = [min(p1.min(), p2.min()) / 2, max(p1.max(), p2.max()) * 2]
    ax.plot(lim, lim, color=Gray, lw=0.8, ls='--')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Sep 2024 - Sep 2025')
    ax.set_ylabel('Sep 2025 - Sep 2026')
    ax.set_title('(3) Year-on-year: stable ranking', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=4, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_b5')


def fig_b6(g, slope, ci, raw, raw_nz, iv, iv_se):
    """Relatia pe grupe (log-log) cu panta 0,5 de referinta si intervalele celor patru estimari ale pantei."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
    ax = axes[0]
    lx, lz = np.log(g['v'].values), np.log(g['z'].values)
    ax.plot(lx, lz, 'o', color=MainBlue, ms=5, label='20 groups of bars by relative volume')
    b0 = np.polyfit(lx, lz, 1)
    xs = np.linspace(lx.min(), lx.max(), 10)
    ax.plot(xs, b0[1] + b0[0] * xs, color=IDAred, lw=1.5, label=f'Fitted slope {slope:.2f}')
    ax.plot(xs, lz.mean() + 0.5 * (xs - lx.mean()), color=Gray, lw=1.0, ls='--', label='Reference slope 0.5')
    ax.set_xlabel('log mean relative volume')
    ax.set_ylabel('log mean normalised |return|')
    ax.set_title('(1) Concave, flatter than a square root', fontsize=9, loc='left')
    ax = axes[1]
    rows = [('Grouped OLS (day bootstrap)', slope, ci[0], ci[1], MainBlue),
            ('Bar-level OLS (clustered)', raw.params.iloc[1], raw.params.iloc[1] - 1.96 * raw.bse.iloc[1],
             raw.params.iloc[1] + 1.96 * raw.bse.iloc[1], IDAred),
            ('Bar-level OLS, r = 0 dropped', raw_nz.params.iloc[1], raw_nz.params.iloc[1] - 1.96 * raw_nz.bse.iloc[1],
             raw_nz.params.iloc[1] + 1.96 * raw_nz.bse.iloc[1], Amber),
            ('IV, lagged same-bar volume', iv, iv - 1.96 * iv_se, iv + 1.96 * iv_se, Purple)]
    for i, (lab, bb, l_, h_, c) in enumerate(rows):
        ax.hlines(i, l_, h_, color=c, lw=2)
        ax.plot(bb, i, 'o', color=c, ms=6)
        ax.text(h_ + 0.012, i, f'{bb:.3f}', va='center', fontsize=7.5, color='black')
    ax.axvline(0.5, color=Gray, ls='--', lw=0.8)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 0.6)
    ax.set_xlabel('Slope with 95% interval (dashed: 0.5)')
    ax.set_title('(2) Four estimates of the slope', fontsize=9, loc='left')
    h, l = axes[0].get_legend_handles_labels()
    fig_legend_bottom(fig, h, l, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_b6')


def fig_b7(b, r1, r2):
    """Profilul orar al mediei |r| (media zilnica pe ora, apoi media pe zile), zile lucratoare si weekend, cu benzi
    bootstrap pe zile; cele doua rapoarte cu intervalele lor."""
    piv = b.assign(hour=b.index.hour).pivot_table(index='date', columns='hour', values='r', aggfunc=lambda v: v.abs().mean())
    we = piv.index.dayofweek >= 5
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3), gridspec_kw=dict(width_ratios=[1.6, 1]))
    ax = axes[0]
    for m, c, lab, sd in [(~we, MainBlue, 'Weekdays', 1), (we, Amber, 'Weekends', 2)]:
        P = piv[m]
        rng = np.random.default_rng(SEED + sd)
        bs = np.array([P.iloc[rng.integers(0, len(P), len(P))].mean().values for _ in range(300)])
        ax.fill_between(piv.columns, 1e4 * np.percentile(bs, 2.5, axis=0), 1e4 * np.percentile(bs, 97.5, axis=0),
                        color='#DADADA', alpha=0.8, lw=0)
        ax.plot(piv.columns, 1e4 * P.mean().values, color=c, lw=1.4, marker='o', ms=3, label=f'{lab}: mean |5-minute return|')
    ax.axvspan(12.6, 16.4, color=IDAred, alpha=0.08, lw=0, label='US morning, 13:00-17:00 UTC')
    ax.axvspan(2.6, 10.4, color=Forest, alpha=0.08, lw=0, label='Asian and European morning, 03:00-11:00 UTC')
    ax.set_xlabel('Hour (UTC)')
    ax.set_ylabel('Mean |return| (bp)')
    ax.set_title('Bitcoin keeps the clock of traditional markets', fontsize=9, loc='left')
    ax = axes[1]
    for i, (lab, (est, lo, hi), c) in enumerate([('US / Asian-European\nmorning (weekdays)', r1, IDAred),
                                                  ('Weekdays / weekends', r2, MainBlue)]):
        ax.hlines(i, lo, hi, color=c, lw=2)
        ax.plot(est, i, 'o', color=c, ms=6)
        ax.text(hi + 0.03, i, f'{est:.2f}', va='center', fontsize=8, color='black')
    ax.axvline(1, color=Gray, ls='--', lw=0.8)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['US / Asian-European\nmorning (weekdays)', 'Weekdays / weekends'], fontsize=8)
    ax.set_ylim(-0.6, 1.6)
    ax.invert_yaxis()
    ax.set_xlim(0.9, 2.2)
    ax.set_xlabel('Ratio of mean |return| with 95% interval')
    ax.set_title('Both ratios exclude 1', fontsize=9, loc='left')
    h, l = axes[0].get_legend_handles_labels()
    fig_legend_bottom(fig, h, l, ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    save_fig('ch10_sem_b7')


def fig_b8(yl, base, d, h1, h2):
    """Preturile rebazate si baza ETF - indice; coeficientii de ajustare si ponderile, cu intervale bootstrap."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2))
    ax = axes[0]
    ax.plot(yl.index, 100 * np.exp(yl['etf'] - yl['etf'].iloc[0]), color=IDAred, lw=1.1, label='Patria-TVBETETF (close)')
    ax.plot(yl.index, 100 * np.exp(yl['idx'] - yl['idx'].iloc[0]), color=MainBlue, lw=1.1, label='BET-TR index')
    ax.set_ylabel('Rebased, first day = 100')
    ax.set_title('(1) Two prices of one basket', fontsize=9, loc='left')
    ax = axes[1]
    bs = 100 * (yl['etf'] - yl['idx'])
    ax.plot(yl.index, bs - bs.mean(), color=Purple, lw=0.9, label='Basis: log ETF - log index, demeaned (%)')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_ylabel('Basis (%)')
    ax.set_title('(2) The basis mean-reverts: cointegration', fontsize=9, loc='left')
    for a in axes:
        a.tick_params(axis='x', labelsize=7.5)
    h1_, l1_ = axes[0].get_legend_handles_labels()
    h2_, l2_ = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1_ + h2_, l1_ + l2_, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    save_fig('ch10_sem_b8_prices')
    ci = lambda k: np.percentile(d[:, k], [2.5, 97.5])
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2))
    ax = axes[0]
    rows = [('ETF, full sample', base['alpha'][0], base['alpha'][0] / base['t'][0], ci(0), IDAred),
            ('Index, full sample', base['alpha'][1], base['alpha'][1] / base['t'][1], ci(1), MainBlue),
            ('ETF, first half', h1['alpha'][0], h1['alpha'][0] / h1['t'][0], None, Amber),
            ('ETF, second half', h2['alpha'][0], h2['alpha'][0] / h2['t'][0], None, Forest)]
    for i, (lab, a, se, c_, col) in enumerate(rows):
        ax.hlines(i - 0.12, a - 1.96 * se, a + 1.96 * se, color=col, lw=2, label='Asymptotic 95% interval' if i == 0 else None)
        if c_ is not None:
            ax.hlines(i + 0.12, c_[0], c_[1], color=col, lw=2, ls=':', label='Block-bootstrap 95% interval' if i == 0 else None)
        ax.plot(a, i - 0.12, 'o', color=col, ms=6)
    ax.axvline(0, color=Gray, ls='--', lw=0.8)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlabel('Adjustment coefficient $\\alpha$')
    ax.set_title('(1) Only the ETF adjusts, and weakly', fontsize=9, loc='left')
    ax = axes[1]
    lo, hi = base['is_lo'][1], base['is_hi'][1]
    ax.barh(0, hi - lo, left=lo, height=0.35, color=Teal, alpha=0.6, label='Index information share: ordering bounds')
    for k, v in [(3, lo), (4, hi)]:
        c_ = ci(k)
        ax.hlines(0.32, c_[0], c_[1], color=MainBlue if k == 3 else IDAred, lw=2,
                  label='Bootstrap 95% interval, lower bound' if k == 3 else 'Bootstrap 95% interval, upper bound')
    ax.plot(base['cs'][1], 1, 'o', color=Purple, ms=6, label='Index component share (point)')
    c_ = ci(2)
    ax.hlines(1, c_[0], c_[1], color=Purple, lw=2, ls=':')
    ax.axvline(0, color=Gray, lw=0.6)
    ax.axvline(1, color=Gray, lw=0.6)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Information share', 'Component share'], fontsize=8)
    ax.set_ylim(-0.5, 1.5)
    ax.invert_yaxis()
    ax.set_xlabel('Share of price discovery attributed to the index')
    ax.set_title('(2) Bounds are wide at daily frequency', fontsize=9, loc='left')
    h1_, l1_ = axes[0].get_legend_handles_labels()
    h2_, l2_ = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1_ + h2_, l1_ + l2_, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.16, 1, 1))
    save_fig('ch10_sem_b8_shares')


def b8_price_discovery(B=B, L=20, seed=SEED):
    """ETF-ul Patria-TVBETETF si indicele BET-TR: VECM cu vectorul (1, -1), limitele ponderii Hasbrouck, ponderea
    Gonzalo-Granger; intervale bootstrap pe blocuri mobile (blocuri de L zile) ale perechilor (X_t, Y_t) din VECM;
    apoi cele doua jumatati ale esantionului."""
    from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order
    y = bet_etf_pair().values
    p = max(int(select_order(y, maxlags=10, deterministic='co').bic), 1)   # BIC pe 1..10 decalaje ale lui dy
    jo = coint_johansen(y, 0, p)                                           # constanta nerestrictionata
    base = price_discovery(y, p)
    dy = np.diff(y, axis=0)
    z = (y[:, 0] - y[:, 1])[:-1]
    X = np.array([[1.0, z[t]] + [v for i in range(1, p + 1) for v in dy[t - i]] for t in range(p, len(dy))])
    Y = dy[p:]
    rng = np.random.default_rng(seed)
    n = len(Y)
    draws = []
    for _ in range(B):
        st = rng.integers(0, n - L + 1, n // L + 1)
        idx = np.concatenate([np.arange(a, a + L) for a in st])[:n]
        Xb, Yb = X[idx], Y[idx]
        Bb = np.linalg.lstsq(Xb, Yb, rcond=None)[0]
        U = Yb - Xb @ Bb
        Om = U.T @ U / (n - X.shape[1])
        al = Bb[1]
        ap = np.array([-al[1], al[0]])
        ISs = []
        for order in ([0, 1], [1, 0]):
            F = np.linalg.cholesky(Om[np.ix_(order, order)])
            v = (ap[order] @ F) ** 2 / (ap @ Om @ ap)
            ISs.append(v[np.argsort(order)][1])
        draws.append((al[0], al[1], ap[1] / ap.sum(), min(ISs), max(ISs)))
    d = np.array(draws)
    ci = lambda k: [float(x) for x in np.percentile(d[:, k], [2.5, 97.5])]
    half = len(y) // 2
    h1, h2 = price_discovery(y[:half], p), price_discovery(y[half:], p)
    fig_b8(bet_etf_pair(), base, d, h1, h2)
    return dict(basis_mean=float(100 * np.mean(y[:, 0] - y[:, 1])), start=bet_etf_pair().index[0].strftime('%Y-%m-%d'),
                end=bet_etf_pair().index[-1].strftime('%Y-%m-%d'), a_etf=float(base['alpha'][0]), a_idx=float(base['alpha'][1]), t_etf=float(base['t'][0]),
                t_idx=float(base['t'][1]), cs_idx=float(base['cs'][1]), is_idx_lo=float(base['is_lo'][1]),
                is_idx_hi=float(base['is_hi'][1]), ci_a_etf=ci(0), ci_a_idx=ci(1), ci_cs_idx=ci(2), ci_is_lo=ci(3),
                ci_is_hi=ci(4), h1_a_etf=float(h1['alpha'][0]), h1_t_etf=float(h1['t'][0]), h2_a_etf=float(h2['alpha'][0]),
                h2_t_etf=float(h2['t'][0]), h1_a_idx=float(h1['alpha'][1]), h2_a_idx=float(h2['alpha'][1]),
                h1_t_idx=float(h1['t'][1]), h2_t_idx=float(h2['t'][1]), corr=base['corr'], n=int(base['n']), p=p,
                trace=float(jo.lr1[0]), cv=float(jo.cvt[0, 1]), trace1=float(jo.lr1[1]), cv1=float(jo.cvt[1, 1]))


def market_illiq(keys, start='2016-09-19'):
    """Ilichiditatea agregata lunara: media transversala a logaritmului Amihud lunar al fiecarui activ."""
    cols = {}
    for k in keys:
        x = pd.concat([returns(k, start).rename('r'), dollar_volume(k, start).rename('dv')], axis=1, join='inner').dropna()
        x = x[x['dv'] > 0]
        m = (1e4 * x['r'].abs() / (x['dv'] / 1e6)).resample('ME').agg(['mean', 'count'])
        cols[k] = np.log(m['mean'].where(m['count'] >= 10))
    return pd.DataFrame(cols).mean(axis=1).rename('lilliq')


def c1_premium():
    out, series = {}, {}
    for name, spec in C_MARKETS.items():
        sym, col = spec['ret']
        p = read_market(sym)[col].loc['2016-08-01':'2026-08-31']
        r = 100 * np.log(p.resample('ME').last()).diff().rename('r')
        li = market_illiq(spec['keys']).dropna()
        # socul de ilichiditate: rezidul unui AR(1) pe logaritmul ilichiditatii
        ar = sm.OLS(li.iloc[1:].values, sm.add_constant(li.shift(1).iloc[1:].values)).fit()
        shock = pd.Series(ar.resid, index=li.index[1:], name='shock')
        d = pd.concat([r, li.shift(1).rename('lag'), shock], axis=1, join='inner').dropna().loc['2016-11':'2026-08']
        fit = sm.OLS(d['r'], sm.add_constant(d[['lag', 'shock']])).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
        # Amihud si Hurvich (2004): regresia augmentata cu reziduul AR(1) calculat cu phi corectat de deplasare,
        # phi_c = phi + (1 + 3 phi) / T + 3 (1 + 3 phi) / T^2; eroarea standard include incertitudinea lui phi_c
        T_ar = len(li) - 1
        phi = ar.params[1]
        phi_c = min(phi + (1 + 3 * phi) / T_ar + 3 * (1 + 3 * phi) / T_ar ** 2, 0.9999)   # trunchiat la 0.9999
        psi_c = np.mean(li.values[1:] - phi_c * li.values[:-1])
        vc = pd.Series(li.values[1:] - psi_c - phi_c * li.values[:-1], index=li.index[1:], name='vc')
        dc = pd.concat([r, li.shift(1).rename('lag'), vc], axis=1, join='inner').dropna().loc['2016-11':'2026-08']
        fc = sm.OLS(dc['r'], sm.add_constant(dc[['lag', 'vc']])).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
        var_phic = (1 + 3 / T_ar + 9 / T_ar ** 2) ** 2 * ar.bse[1] ** 2
        se_c = float(np.sqrt(fc.params['vc'] ** 2 * var_phic + fc.bse['lag'] ** 2))
        ah = dict(phi_c=float(phi_c), b_lag_c=float(fc.params['lag']), se_lag_c=se_c, t_lag_c=float(fc.params['lag'] / se_c),
                  bias=float(fit.params['lag'] - fc.params['lag']))
        out[name] = dict(n=int(len(d)), b_lag=float(fit.params['lag']), se_lag=float(fit.bse['lag']),
                         t_lag=float(fit.tvalues['lag']), b_shock=float(fit.params['shock']),
                         se_shock=float(fit.bse['shock']), t_shock=float(fit.tvalues['shock']), r2=float(fit.rsquared),
                         phi=float(ar.params[1]), start=d.index[0].strftime('%Y-%m'), end=d.index[-1].strftime('%Y-%m'), **ah)
        series[name] = d
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
    for ax, (name, d) in zip(axes, series.items()):
        c = GROUP_COL[name]
        ax.scatter(d['shock'], d['r'], s=12, color=c, label=f'{name}: monthly observations')
        xs = np.linspace(d['shock'].min(), d['shock'].max(), 20)
        f = sm.OLS(d['r'], sm.add_constant(d['shock'])).fit()
        ax.plot(xs, f.params.iloc[0] + f.params.iloc[1] * xs, color='black', lw=1.0)
        ax.axhline(0, color=Gray, lw=0.5)
        ax.set_xlabel('Illiquidity shock (AR(1) residual of log illiquidity)')
        ax.set_title(name, fontsize=9, loc='left', color=c)
    axes[0].set_ylabel('Market return in the same month (%)')
    h = [a.get_legend_handles_labels()[0][0] for a in axes]
    fig_legend_bottom(fig, h, [a.get_legend_handles_labels()[1][0] for a in axes] , ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_premium')
    # sectiunea transversala BVB: portofoliul celor mai putin lichide vs cele mai lichide (reechilibrare anuala)
    keys = C_MARKETS['BVB']['keys']
    px = pd.concat({k: read_market(ASSETS[k][0])['adjusted_close'] for k in keys}, axis=1).loc['2017-01-01':'2026-08-31']
    mret = px.resample('ME').last().pct_change(fill_method=None)       # randamente simple totale (portofoliu exact)
    spreads = []
    for y in range(2018, 2027):
        il = {}
        for k in keys:
            rr = returns(k, f'{y - 1}-01-01', f'{y - 1}-12-31')
            dv = dollar_volume(k, f'{y - 1}-01-01', f'{y - 1}-12-31')
            if len(rr) > 150:
                il[k] = float((rr.abs() / dv).dropna().mean())
        if len(il) < 6:
            continue
        srt = sorted(il, key=il.get)
        liq, ill = srt[:len(srt) // 2], srt[-(len(srt) // 2):]
        yr = mret.loc[str(y)]
        spreads.append((yr[ill].mean(axis=1) - yr[liq].mean(axis=1)).dropna())
    ls = 100 * pd.concat(spreads)
    tt = sm.OLS(ls, np.ones(len(ls))).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
    out['bvb_ls'] = dict(mean=float(ls.mean()), se=float(tt.bse.iloc[0]), t=float(tt.tvalues.iloc[0]), n=int(len(ls)),
                         ann=float(12 * ls.mean()), start=ls.index[0].strftime('%Y-%m'), end=ls.index[-1].strftime('%Y-%m'))
    fig_c1(out, ls)
    return out


def fig_c1(out, ls):
    """Coeficientul ilichiditatii decalate (HAC si cu deplasare redusa) si seria lunara nelichid minus lichid (BVB)."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3), gridspec_kw=dict(width_ratios=[1, 1.4]))
    ax = axes[0]
    for i, name in enumerate(('BVB', 'US', 'Crypto')):
        o = out[name]
        c = GROUP_COL[name]
        ax.hlines(i - 0.13, o['b_lag'] - 1.96 * o['se_lag'], o['b_lag'] + 1.96 * o['se_lag'], color=c, lw=2,
                  label='OLS with HAC SE' if i == 0 else None)
        ax.plot(o['b_lag'], i - 0.13, 'o', color=c, ms=6)
        ax.hlines(i + 0.13, o['b_lag_c'] - 1.96 * o['se_lag_c'], o['b_lag_c'] + 1.96 * o['se_lag_c'], color=c, lw=2,
                  ls=':', label='Reduced-bias (Amihud-Hurvich)' if i == 0 else None)
        ax.plot(o['b_lag_c'], i + 0.13, 's', color=c, ms=5)
    ax.axvline(0, color=Gray, ls='--', lw=0.8)
    ax.set_yticks(range(3))
    ax.set_yticklabels(['BVB', 'US', 'Crypto'])
    ax.invert_yaxis()
    ax.set_xlabel('Coefficient of lagged illiquidity, 95% interval')
    ax.set_title('(1) Predictive effect: crypto only, and fragile', fontsize=9, loc='left')
    ax = axes[1]
    ax.bar(ls.index, ls.values, width=20, color=[IDAred if v > 0 else MainBlue for v in ls.values],
           label='BVB illiquid-minus-liquid return, monthly (%)')
    ax.axhline(ls.mean(), color='black', lw=1.0, ls='--', label=f'Mean {ls.mean():.2f}% per month')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_ylabel('Return difference (%)')
    ax.set_title('(2) Annual sorts of the ten blue chips', fontsize=9, loc='left')
    ax.tick_params(axis='x', labelsize=7.5)
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.13, 1, 1))
    save_fig('ch10_sem_c1')


def fig_a1_bounce(c=0.09, sig2=0.0358, n_path=40, n_long=20000, seed=SEED):
    """A1 simulat: pretul eficient, pretul tranzactiilor si autocorelatia variatiilor, cu parametrii din A1(b)."""
    rng = np.random.default_rng(seed)
    m = 20 + np.cumsum(rng.normal(0, np.sqrt(sig2), n_long))
    q = np.where(rng.random(n_long) < 0.5, 1, -1)
    p = m + c * q
    dp = np.diff(p)
    dm = dp - dp.mean()
    acf = [float(np.mean(dm[k:] * dm[:-k]) / np.mean(dm ** 2)) for k in range(1, 9)]
    rho = -c ** 2 / (sig2 + 2 * c ** 2)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw=dict(width_ratios=[1.5, 1]))
    ax = axes[0]
    k = np.arange(n_path)
    ax.plot(k, m[:n_path], color=MainBlue, lw=1.4, label='Efficient price $m_t$')
    ax.plot(k, p[:n_path], color=IDAred, lw=0.9, marker='o', ms=3, label='Trade price $p_t = m_t + c\\,q_t$, c = 0.09 lei')
    ax.set_xlabel('Trade number')
    ax.set_ylabel('Price (lei)')
    ax.set_title('Simulated: trades alternate between bid and ask', fontsize=9, loc='left')
    ax = axes[1]
    lags = np.arange(1, 9)
    ax.bar(lags, acf, color=IDAred, width=0.6, label=f'Sample autocorrelation of $\\Delta p_t$ ({n_long:,} trades)')
    ax.axhline(rho, color=MainBlue, ls='--', lw=1.1, label=f'Theory at lag 1: $-c^2/(\\sigma^2 + 2c^2)$ = {rho:.3f}')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.set_title('Only lag 1 is negative', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    save_fig('ch10_sem_a1_bounce')
    return dict(acf1=acf[0], rho=rho)


def fig_a2_sampling(var=0.0520, cov=-0.0081, Ts=(78, 250), R=20000):
    """A2: distributia de selectie a autocovariantei (simulata sub modelul Roll, semne binare, demediere) si
    aproximarea Normala a lui Bartlett; coada pozitiva hasurata."""
    from scipy.stats import norm
    c = np.sqrt(-cov)
    sig = np.sqrt(var - 2 * c ** 2)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), sharey=False)
    out = {}
    for ax, T, col in zip(axes, Ts, (IDAred, MainBlue)):
        g = roll_mc(c, sig, T=T + 1, R=R, seed=SEED)        # T variatii de pret
        se = np.sqrt((var ** 2 + 3 * cov ** 2) / T)
        ax.hist(g, bins=60, density=True, color=col, alpha=0.45, label='Simulated under Roll (binary signs, demeaned)')
        xs = np.linspace(cov - 4.5 * se, cov + 4.5 * se, 300)
        ax.plot(xs, norm.pdf(xs, cov, se), color='black', lw=1.1, label='Normal approximation $N(\\gamma_1, (\\gamma_0^2 + 3\\gamma_1^2)/T)$')
        xp = xs[xs > 0]
        ax.fill_between(xp, 0, norm.pdf(xp, cov, se), color=Amber, alpha=0.6, lw=0, label='Positive tail: $\\hat\\gamma_1 > 0$')
        ax.axvline(cov, color=Gray, ls='--', lw=0.8)
        ax.set_xlabel('Sample autocovariance $\\hat\\gamma_1$ (lei$^2$)')
        ax.set_title(f'T = {T} price changes: P(positive) {100 * norm.cdf(cov / se):.1f}% (Normal), {100 * (g > 0).mean():.1f}% (simulated)',
                     fontsize=8.5, loc='left')
        out[T] = dict(p_sim=float((g > 0).mean()), mean_sim=float(g.mean()), sd_sim=float(g.std()),
                      bias_formula=float(-(var + 2 * cov) / T))
    axes[0].set_ylabel('Density')
    h, l = axes[0].get_legend_handles_labels()
    fig_legend_bottom(fig, h, l, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    save_fig('ch10_sem_a2_sampling')
    return out


def fig_a3_spread(dv=20.0):
    """A3: spread-ul Glosten-Milgrom in functie de probabilitatea initiala theta, pentru trei ponderi mu."""
    th = np.linspace(0.005, 0.995, 200)
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    for mu, c in [(0.1, Amber), (0.3, IDAred), (0.5, MainBlue)]:
        ax.plot(th, gm_spread_formula(th, mu, dv), color=c, lw=1.5, label=f'$\\mu$ = {mu}: maximum $\\mu\\Delta V$ = {mu * dv:.0f} at $\\theta$ = 1/2')
    for t_, lab in [(0.5, 'prior'), (0.65, 'after one buy'), (0.7, '$\\theta$ = 0.7')]:
        v = gm_spread_formula(t_, 0.3, dv)
        ax.plot(t_, v, 'o', color='black', ms=4)
        ax.annotate(f'{lab}: {v:.2f}', (t_, v), xytext=(6, 4) if t_ != 0.65 else (-75, -14), textcoords='offset points',
                    fontsize=7.5, color='black')
    ax.set_xlabel('Probability of the high value, $\\theta$')
    ax.set_ylabel('Spread $a - b$ ($V_L$ = 90, $V_H$ = 110)')
    legend_outside_bottom(ax, ncol=2, y=-0.22)
    plt.tight_layout()
    save_fig('ch10_sem_a3_spread')


def fig_a4_learning(mus=(0.1, 0.3, 0.5), level=108.0, vl=90.0, vh=110.0, kmax=14):
    """A4: bid-ul dupa k cumparari consecutive (scenariu conditionat) si traiectoriile simulate din curs."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
    ax = axes[0]
    for mu, c in zip(mus, (Amber, IDAred, MainBlue)):
        theta, bids = 0.5, []
        for k in range(kmax + 1):
            ask, bid, tb, ts = gm_quotes(theta, mu, vl, vh)
            bids.append(bid)
            theta = tb
        bids = np.array(bids)
        k1 = int(np.argmax(bids > level))
        ax.plot(range(kmax + 1), bids, color=c, lw=1.4, marker='o', ms=3, label=f'$\\mu$ = {mu}: bid > 108 after k = {k1} buys')
    ax.axhline(level, color=Gray, ls='--', lw=0.8)
    ax.set_xlabel('Number of consecutive buys k')
    ax.set_ylabel('Bid quoted after k buys')
    ax.set_title('(1) Conditioned scenario: every trade is a buy', fontsize=9, loc='left')
    ax = axes[1]
    for mu, c in [(0.1, Amber), (0.3, IDAred)]:
        d = gm_simulate(mu, n=60, seed=3)
        ax.plot(d['t'], d['bid'], color=c, lw=1.2, ls='--', label=f'Lecture path, $\\mu$ = {mu}: bid')
    ax.axhline(level, color=Gray, ls='--', lw=0.8)
    ax.set_xlabel('Trade number')
    ax.set_ylabel('Bid')
    ax.set_title('(2) Simulated order flow with uninformed sells', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.12, 1, 1))
    save_fig('ch10_sem_a4_learning')


def fig_a6_kyle(sigma_v=2.0, sigma_u=10_000.0, p0=50.0, n=200_000, seed=SEED, n_show=3000):
    """A6: lambda ca panta de regresie (simularea din A6) si raportul de tip Amihud cu doua numitoare."""
    rng = np.random.default_rng(seed)
    k = kyle(sigma_v, sigma_u)
    v = p0 + sigma_v * rng.standard_normal(n)
    x = k['beta'] * (v - p0)
    u = sigma_u * rng.standard_normal(n)
    y = x + u
    r1 = a6_kyle_ols(sigma_v, sigma_u, p0, n, seed)
    r2 = a6_kyle_ols(sigma_v, 2 * sigma_u, p0, n, seed)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
    ax = axes[0]
    ax.scatter(y[:n_show] / 1e3, v[:n_show] - p0, s=4, color=MainBlue, alpha=0.4, label='$v - p_0$ (first 3,000 auctions)')
    ys = np.linspace(y.min(), y.max(), 10)
    ax.plot(ys / 1e3, k['lam'] * ys, color=IDAred, lw=1.6, label=f'$p - p_0 = \\lambda y$, $\\lambda$ = {k["lam"]:.4f} USD per share')
    ax.set_xlim(-45, 45)
    ax.set_xlabel('Net order flow $y$ (thousand shares)')
    ax.set_ylabel('USD')
    ax.set_title('(1) Both regressions on $y$ have slope $\\lambda$', fontsize=9, loc='left')
    ax = axes[1]
    labs = ['Theory $\\lambda/p_0^2$', 'Net flow $|y|$', 'Gross proxy $|x| + |u|$']
    xs = np.arange(3)
    w = 0.36
    v1 = [r1['target'] * 1e10, r1['am_net'] * 1e10, r1['am_vol'] * 1e10]
    v2 = [r2['target'] * 1e10, r2['am_net'] * 1e10, r2['am_vol'] * 1e10]
    b1 = ax.bar(xs - w / 2, v1, w, color=MainBlue, label='$\\sigma_u$ = 10,000 shares')
    b2 = ax.bar(xs + w / 2, v2, w, color=Amber, label='$\\sigma_u$ = 20,000 shares (doubled noise)')
    for bars in (b1, b2):
        for bb in bars:
            ax.text(bb.get_x() + bb.get_width() / 2, bb.get_height() + 6, f'{bb.get_height():.0f}', ha='center', fontsize=7.5, color='black')
    ax.set_xticks(xs)
    ax.set_xticklabels(labs, fontsize=8)
    ax.set_ylabel('bp per USD 1 million')
    ax.set_title('(2) Amihud-type ratio: the denominator matters', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    save_fig('ch10_sem_a6_kyle')
    return dict(target2=r2['target'] * 1e10, amnet2=r2['am_net'] * 1e10, amvol2=r2['am_vol'] * 1e10, ratio2=r2['am_ratio'])


def fig_ac_seminar():
    """A7-A8: detinerile x_k dupa fiecare zi (discret, N = 5) si solutia continua; frontiera cost - risc."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
    ax = axes[0]
    tt = np.linspace(0, AC['T'], 200)
    pts = {}
    for lam, c, lab in [(0, MainBlue, 'Even selling, $\\lambda$ = 0'), (2e-6, Forest, 'A7: $\\lambda$ = 2e-6'),
                        (2e-5, Orange, 'A8: $\\lambda$ = 2e-5')]:
        t, x, E, V = ac_trajectory(lam=lam, **AC)
        pts[lam] = (E, np.sqrt(V))
        ax.plot(t, x / 1e3, color=c, lw=0, marker='o', ms=5, label=lab + ': holdings $x_k$ after day k')
        if lam > 0:
            kc = np.sqrt(lam * AC['sigma'] ** 2 / AC['eta'])
            ax.plot(tt, AC['X'] * np.sinh(kc * (AC['T'] - tt)) / np.sinh(kc * AC['T']) / 1e3, color=c, lw=1, ls='--')
        else:
            ax.plot(tt, AC['X'] * (1 - tt / AC['T']) / 1e3, color=c, lw=1, ls='--')
    ax.set_xlabel('Day k')
    ax.set_ylabel('Shares still held (thousands)')
    ax.set_title('(1) Discrete schedule (dots), continuous solution (dashed)', fontsize=9, loc='left')
    ax = axes[1]
    fr = ac_frontier(lams=np.logspace(-8, -3.5, 60), **AC)
    ax.plot(fr['sd'] / 1e6, fr['E'] / 1e6, color=Purple, lw=1.5, label='Efficient frontier (N = 5 days)')
    for lam, c in [(0, MainBlue), (2e-6, Forest), (2e-5, Orange)]:
        E, sd = pts[lam]
        ax.plot(sd / 1e6, E / 1e6, 'o', color=c, ms=7)
        ax.annotate(f'({sd / 1e6:.2f}, {E / 1e6:.2f})', (sd / 1e6, E / 1e6), xytext=(6, 3), textcoords='offset points',
                    fontsize=7.5, color='black')
    ax.set_xlabel('Standard deviation of cost (USD million)')
    ax.set_ylabel('Expected cost (USD million)')
    ax.set_title('(2) Cost versus risk', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=2, y=0.02)
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    save_fig('ch10_sem_ac')


def fig_pin(**par):
    """A9-A10: cele trei stari ale modelului EKOP (centrele Poisson) si probabilitatile a posteriori pentru patru zile."""
    par = par or PIN_PAR
    a, d_, mu, eb, es = (par[k] for k in ('alpha', 'delta', 'mu', 'eb', 'es'))
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), gridspec_kw=dict(width_ratios=[1, 1.5]))
    ax = axes[0]
    th = np.linspace(0, 2 * np.pi, 100)
    for (bx, sx), lab, c in [((eb, es), f'No news, prob. {1 - a:.2f}', MainBlue),
                             ((eb, es + mu), f'Bad news, prob. {a * d_:.2f}', IDAred),
                             ((eb + mu, es), f'Good news, prob. {a * (1 - d_):.2f}', Forest)]:
        ax.plot(bx + 2 * np.sqrt(bx) * np.cos(th), sx + 2 * np.sqrt(sx) * np.sin(th), color=c, lw=1.3, label=lab)
        ax.plot(bx, sx, '+', color=c, ms=8)
    for lab, B_, S_ in PIN_DAYS:
        ax.plot(B_, S_, 'o', color='black', ms=4)
        ax.annotate(lab.split(':')[0], (B_, S_), xytext=(4, 4), textcoords='offset points', fontsize=7, color='black')
    ax.set_xlabel('Buys B per day')
    ax.set_ylabel('Sells S per day')
    ax.set_title('(1) States: Poisson centres, 2 SD circles', fontsize=9, loc='left')
    ax = axes[1]
    post = {}
    xs = np.arange(len(PIN_DAYS))
    w = 0.26
    for j, (lab_, c) in enumerate([('No news', MainBlue), ('Bad news', IDAred), ('Good news', Forest)]):
        vals = []
        for lab, B_, S_ in PIN_DAYS:
            pp = pin_posterior(B_, S_, a, d_, mu, eb, es)
            post[lab] = [float(v) for v in pp]
            vals.append(pp[j])
        ax.bar(xs + (j - 1) * w, vals, w, color=c, label=f'Posterior: {lab_.lower()}')
    ax.set_xticks(xs)
    ax.set_xticklabels([l.split(': ')[1] for l, _, _ in PIN_DAYS], fontsize=7.5)
    ax.set_ylabel('Posterior probability')
    ax.set_title('(2) Posterior probability of each state', fontsize=9, loc='left')
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    fig_legend_bottom(fig, h1 + h2, l1 + l2, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.13, 1, 1))
    save_fig('ch10_sem_pin')
    return post


def fig_c2_roundtrip(mid=20.0, s=0.05):
    """C2: bid, pret de mijloc si ask; o jumatate de spread pe fiecare sens, un spread pentru dus-intors."""
    a, b = mid + s / 2, mid - s / 2
    fig, ax = plt.subplots(figsize=(7.5, 3.0))
    for y, lab, c in [(a, f'Ask a = {a:.3f} lei: you buy here', IDAred), (mid, f'Mid m = {mid:.3f} lei', Gray),
                      (b, f'Bid b = {b:.3f} lei: you sell here', MainBlue)]:
        ax.hlines(y, 0, 1, color=c, lw=2 if c != Gray else 1, ls='-' if c != Gray else '--')
        ax.text(1.03, y, lab, va='center', fontsize=8.5, color='black')
    ax.annotate('', xy=(0.25, a), xytext=(0.25, mid), arrowprops=dict(arrowstyle='<->', color=IDAred, lw=1.2))
    ax.text(0.27, (a + mid) / 2, f'buy leg: c = s/2 = {s / 2:.3f} lei ({100 * s / 2 / mid:.3f}%)', va='center', fontsize=8, color='black')
    ax.annotate('', xy=(0.25, b), xytext=(0.25, mid), arrowprops=dict(arrowstyle='<->', color=MainBlue, lw=1.2))
    ax.text(0.27, (b + mid) / 2, f'sell leg: c = {s / 2:.3f} lei ({100 * s / 2 / mid:.3f}%)', va='center', fontsize=8, color='black')
    ax.annotate('', xy=(0.8, a), xytext=(0.8, b), arrowprops=dict(arrowstyle='<->', color=Purple, lw=1.6))
    ax.text(0.83, mid + 0.012, f'round trip:\na - b = s = {s:.2f} lei\n= {100 * s / mid:.2f}% of the price', fontsize=8,
            color='black', va='center')
    ax.set_xlim(0, 1.6)
    ax.set_ylim(b - 0.008, a + 0.012)
    ax.set_xticks([])
    ax.set_ylabel('Price (lei)')
    ax.spines['bottom'].set_visible(False)
    plt.tight_layout()
    save_fig('ch10_sem_c2_roundtrip')


def c3_race(B=B, L=20, seed=SEED):
    """V_t, sigma_t pe zile; mediile si corelatia anuala cu intervale bootstrap (zile independente si blocuri mobile
    de L zile); premiul anual; taxa zilnica la valoarea medie si raportul efectiv Pi_t / V_t."""
    d = race_daily(intraday_spy())
    P = race_prize(d)
    d['ratio6'] = ABO['col6_v'] + ABO['col6_s'] * d['sigma'] * d.groupby('year')['V'].transform('mean') / d['V']
    years = {}
    for y, g in d.groupby('year'):
        v, sg = g['V'].values, g['sigma'].values
        n = len(g)
        rng = np.random.default_rng(seed)
        iid = [np.corrcoef(v[i], sg[i])[0, 1] for i in (rng.integers(0, n, n) for _ in range(B))]
        rng = np.random.default_rng(seed)
        blk = []
        for _ in range(B):
            st = rng.integers(0, n - L + 1, n // L + 1)
            i = np.concatenate([np.arange(a_, a_ + L) for a_ in st])[:n]
            blk.append(np.corrcoef(v[i], sg[i])[0, 1])
        years[int(y)] = dict(V_mean=float(v.mean() / 1e9), sigma_mean=float(sg.mean()), corr=float(np.corrcoef(v, sg)[0, 1]),
                             ci_iid=[float(x) for x in np.percentile(iid, [2.5, 97.5])],
                             ci_blk=[float(x) for x in np.percentile(blk, [2.5, 97.5])], n=int(n), **{k: P[y][k] for k in ('col2', 'col6', 'lo', 'hi')})
    top = d['tax6'].sort_values(ascending=False)
    rtop = d['ratio6'].sort_values(ascending=False)
    a7 = pd.Timestamp('2025-04-07')
    fig, ax = plt.subplots(figsize=(10, 3.3))
    ax.plot(d.index, d['tax6'], color=MainBlue, lw=0.8, label='Tax at mean traded value: 0.3354 + 0.0066 $\\sigma_t$ (bp)')
    ax.plot(d.index, d['ratio6'], color=IDAred, lw=0.8, label='Realised ratio $\\Pi_t / V_t$ = 0.3354 + 0.0066 $\\sigma_t \\bar V / V_t$ (bp)')
    ax.axhline(ABO['hi'], color=Forest, lw=0.9, ls=':', label='Highest Table XIV scenario, 0.74 bp')
    ax.annotate(f"7 April 2025: {d.loc[a7, 'tax6']:.2f} bp at mean value,\n{d.loc[a7, 'ratio6']:.2f} bp realised (traded value USD {d.loc[a7, 'V'] / 1e9:.0f} billion)",
                xy=(a7, d.loc[a7, 'tax6']), xytext=(pd.Timestamp('2022-09-01'), 0.95), fontsize=7.5, color='black',
                arrowprops=dict(arrowstyle='->', color='black', lw=0.7))
    ax.set_ylabel('Implied tax (bp of traded value)')
    ax.set_ylim(0.1, 1.15)
    legend_outside_bottom(ax, ncol=2, y=-0.12)
    plt.tight_layout()
    save_fig('ch10_sem_c3_tax')
    return dict(years=years, top5=[(str(t.date()), float(top[t])) for t in top.index[:5]],
                ratio_max=(str(rtop.index[0].date()), float(rtop.iloc[0])), ratio_a7=float(d.loc[a7, 'ratio6']),
                V_a7=float(d.loc[a7, 'V'] / 1e9), tax_a7=float(d.loc[a7, 'tax6']), sigma_a7=float(d.loc[a7, 'sigma']),
                n_above_hi_ratio=int((d['ratio6'] > ABO['hi']).sum()), tax_median=float(d['tax6'].median()),
                thr=float((ABO['col2'] - ABO['col6_v']) / ABO['col6_s']))

## Run

In [ ]:
for f in [a1_roll, a2_roll_se, a3_gm, a4_gm_learning, a5_kyle, a6_kyle_ols, a7_ac, a8_ac, a9_pin, a10_pin]:
    print(f.__name__, f())
print('B1', b1_ushape())
print('B2', b2_spreads())
print('B3', {k: v for k, v in b3_cross().items() if k != 'table'})
print('B2 (d)', roll_small_sample(intraday_spreads(intraday_spy())))
print('B4', b4_illiq_vix())
print('B5', b5_amihud_groups())
print('B6', b6_sqrt())
print('B7', b7_btc())
print('B8', b8_price_discovery())
print('C1', c1_premium())
print(fig_a1_bounce(), fig_a2_sampling())
fig_a3_spread()
fig_a4_learning()
print(fig_a6_kyle())
fig_ac_seminar()
print(fig_pin())
print(b2_mc(intraday_spreads(intraday_spy())))
fig_c2_roundtrip()
print(c3_race())

![ch10_sem_ushape](./ch10_sem_ushape.png)

Output: `ch10_sem_ushape.pdf`

![ch10_sem_cross](./ch10_sem_cross.png)

Output: `ch10_sem_cross.pdf`

![ch10_sem_premium](./ch10_sem_premium.png)

Output: `ch10_sem_premium.pdf`

![ch10_sem_a1_bounce](./ch10_sem_a1_bounce.png)

Output: `ch10_sem_a1_bounce.pdf`

![ch10_sem_a2_sampling](./ch10_sem_a2_sampling.png)

Output: `ch10_sem_a2_sampling.pdf`

![ch10_sem_a3_spread](./ch10_sem_a3_spread.png)

Output: `ch10_sem_a3_spread.pdf`

![ch10_sem_a4_learning](./ch10_sem_a4_learning.png)

Output: `ch10_sem_a4_learning.pdf`

![ch10_sem_a6_kyle](./ch10_sem_a6_kyle.png)

Output: `ch10_sem_a6_kyle.pdf`

![ch10_sem_ac](./ch10_sem_ac.png)

Output: `ch10_sem_ac.pdf`

![ch10_sem_pin](./ch10_sem_pin.png)

Output: `ch10_sem_pin.pdf`

![ch10_sem_b2_estimates](./ch10_sem_b2_estimates.png)

Output: `ch10_sem_b2_estimates.pdf`

![ch10_sem_b2_mc](./ch10_sem_b2_mc.png)

Output: `ch10_sem_b2_mc.pdf`

![ch10_sem_b4](./ch10_sem_b4.png)

Output: `ch10_sem_b4.pdf`

![ch10_sem_b5](./ch10_sem_b5.png)

Output: `ch10_sem_b5.pdf`

![ch10_sem_b6](./ch10_sem_b6.png)

Output: `ch10_sem_b6.pdf`

![ch10_sem_b7](./ch10_sem_b7.png)

Output: `ch10_sem_b7.pdf`

![ch10_sem_b8_prices](./ch10_sem_b8_prices.png)

Output: `ch10_sem_b8_prices.pdf`

![ch10_sem_b8_shares](./ch10_sem_b8_shares.png)

Output: `ch10_sem_b8_shares.pdf`

![ch10_sem_c1](./ch10_sem_c1.png)

Output: `ch10_sem_c1.pdf`

![ch10_sem_c2_roundtrip](./ch10_sem_c2_roundtrip.png)

Output: `ch10_sem_c2_roundtrip.pdf`

![ch10_sem_c3_tax](./ch10_sem_c3_tax.png)

Output: `ch10_sem_c3_tax.pdf`